# Audiolivro com leitura guiada: Memórias do Subsolo

Fiódor Dostoiévski (1864, domínio público). Fonte: Записки из подполья, Часть I, Викитека (ru.wikisource.org), texto da revista Эпоха, 1864. Tradução nossa do russo, conferida com a tradução inglesa de Constance Garnett (1918, domínio público).

Duas vozes, nunca trocadas: o **leitor** lê só o texto do autor, e o **comentarista** entra entre os trechos com contexto, ligações dentro da obra e com outras obras. Cada corte é anunciado pelo leitor entre parênteses.

**Antes de começar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois rode as células de cima para baixo. Rodar de novo qualquer célula é seguro.

**Saída** (no seu Google Drive, em `MyDrive/Audiolivros-pessoais/subsolo/capNN/`), para cada capítulo:
- `capNN_guiada.mp3`: texto e comentários, com um sinal curto antes e depois de cada comentário;
- `capNN_limpa.mp3`: só o texto do autor, para reouvir sem comentários.

**Cortes deste lote:**
- Parte I, cap. I: os gritos contra os velhos respeitáveis (1 frase).
- Parte I, cap. I: a criada e os conselhos para sair de Petersburgo (2 frases).
- Parte I, cap. II: as leis da consciência intensificada e o consolo do canalha (2 frases).
- Parte I, cap. III: o rato que se vê como rato sem que ninguém peça (2 frases).
- Parte I, cap. V: as leis da natureza não podiam ser culpadas por isso (2 frases).
- Parte I, cap. VII: o amigo que fala pela razão e age contra ela (1 parágrafo encurtado).
- Parte I, cap. VII: o exemplo de Cleópatra e dos alfinetes de ouro (3 frases).
- Parte I, cap. VIII: a figa prevista numa tabela e a aceitação da retorta (meio parágrafo).
- Parte I, cap. VIII: o querer que às vezes combina com a razão (3 frases).
- Parte I, cap. VIII: as piadas sobre o Colosso de Rodes, os uniformes e os sábios que acabam fazendo escândalo (6 frases).
- Parte I, cap. XI: o aviso de escrever sem ordem e a discussão sobre por que se explica (5 frases).

**Licença:** a voz padrão (OmniVoice) é de uso **não comercial** (CC-BY-NC). As vozes são sintéticas, sem clonar ninguém.

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("ATENÇÃO: sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
          "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive e pastas ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

CAPITULOS_PARA_GERAR = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]   # quais capítulos narrar (ex.: [2])

LIVRO = "/content/drive/MyDrive/Audiolivros-pessoais/subsolo"
PASTA_VOZES = os.path.join(LIVRO, "vozes")
os.makedirs(PASTA_VOZES, exist_ok=True)
print("Os arquivos vão para:", LIVRO)

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min) ──────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="instalação do backend")

from huggingface_hub import snapshot_download
print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
snapshot_download("k2-fsa/OmniVoice")
print("Pronto.")

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4.5. Aceitar as licenças dos modelos (obrigatório antes de gerar) ───────
# Sem este passo o /generate falha. Rodar esta célula aceita as licenças em seu nome:
# OmniVoice é CC-BY-NC (uso NÃO comercial: o áudio não pode ser vendido).
import requests
BASE = "http://127.0.0.1:3900"

for repo in ("k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"):
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    if st.get("accepted"):
        print(f"[ok] {repo} já aceito")
        continue
    r = requests.post(
        f"{BASE}/models/licenses/accept",
        json={"repo_id": repo, "fingerprint": st["fingerprint"], "accepted": True},
        timeout=30,
    )
    r.raise_for_status()
    print(f"[aceito] {repo} ({st.get('license')})")

## Roteiro

Cada trecho é `[voz, texto, velocidade, pausa depois em segundos]`, com voz `leitor` ou `comentarista`.

- O **leitor** só lê frases do autor. Um corte aparece como `(Corte: ...)` e fica nas duas versões.
- O **comentarista** só entra entre trechos, nunca no meio de uma frase do autor.
- **voz** aceita só estes termos: `male`/`female`; `child`, `teenager`, `young adult`, `middle-aged`, `elderly`; `very low pitch` … `very high pitch`; `whisper`. Se não gostar de uma voz, troque a `semente`.

Edite à vontade. Os trechos que você mudar são gerados de novo na célula 7.

In [ ]:
# ── 5. Roteiro (EDITE AQUI) ─────────────────────────────────────────────────
VOZES = {'leitor': {'voz': 'male, middle-aged, low pitch', 'semente': 601},
 'comentarista': {'voz': 'female, young adult, moderate pitch', 'semente': 611}}

CAPITULOS = {
    1: {"titulo": "O homem doente", "trechos": [
        ["comentarista", "Memórias do Subsolo, de Fiódor Dostoiévski, em leitura guiada. Parte um, capítulo um.", 1.0, 0.8],
        ["comentarista", "Eu sou a voz dos comentários. A outra voz, a do leitor, só lê o texto de Dostoiévski, numa tradução nossa, feita do russo. Quando houver um corte, o leitor avisa entre parênteses.", 1.0, 0.5],
        ["comentarista", "O livro saiu em 1864, na revista Época, dos irmãos Dostoiévski. Foi um ano duro para ele: a primeira mulher morreu em abril, e o irmão Mikhail, em julho.", 1.0, 0.5],
        ["comentarista", "Antes do primeiro capítulo, há uma nota do próprio autor. Ela importa, porque avisa que quem fala no livro não é Dostoiévski.", 1.0, 1.0],
        ["leitor", "Nota do autor.", 0.95, 0.6],
        ["leitor", "Tanto o autor destas memórias quanto as próprias memórias são, evidentemente, inventados. Mesmo assim, pessoas como o autor de memórias assim não só podem como até devem existir na nossa sociedade,", 0.95, 0.5],
        ["leitor", "se levarmos em conta as circunstâncias em que a nossa sociedade, de modo geral, se formou.", 0.95, 0.5],
        ["leitor", "Eu quis mostrar ao público, com mais destaque do que de costume, um dos caracteres de um tempo que acabou de passar. É um dos representantes de uma geração que ainda vive os seus últimos dias.", 0.95, 0.5],
        ["leitor", "Neste fragmento, intitulado O subsolo, essa pessoa apresenta a si mesma e ao seu modo de ver, e como que quer explicar as causas pelas quais ela surgiu, e tinha de surgir, no nosso meio.", 0.95, 0.5],
        ["leitor", "No fragmento seguinte virão as verdadeiras memórias dessa pessoa, sobre alguns acontecimentos da sua vida. Fiódor Dostoiévski.", 0.95, 1.2],
        ["leitor", "Parte um. O subsolo. Capítulo um.", 0.95, 1.0],
        ["leitor", "Sou um homem doente... Sou um homem mau. Um homem desagradável, é o que eu sou. Acho que estou doente do fígado.", 0.95, 0.6],
        ["leitor", "Aliás, não entendo patavina da minha doença, e não sei ao certo o que me dói. Não me trato e nunca me tratei, embora respeite a medicina e os médicos.", 0.95, 0.5],
        ["leitor", "Além disso, sou supersticioso ao extremo; bem, pelo menos o bastante para respeitar a medicina. (Sou instruído o bastante para não ser supersticioso, mas sou supersticioso.)", 0.95, 0.5],
        ["leitor", "Não, senhores, eu não quero me tratar por raiva. Isso, com certeza, os senhores não vão ter a bondade de entender. Pois eu entendo.", 0.95, 0.5],
        ["leitor", "Eu, naturalmente, não saberia explicar a quem exatamente vou atingir com a minha raiva. Sei perfeitamente que não vou aprontar nada contra os médicos por não me tratar com eles;", 0.95, 0.5],
        ["leitor", "sei melhor do que ninguém que, com tudo isso, só vou prejudicar a mim mesmo, e a mais ninguém. Mas, mesmo assim, se não me trato, é por raiva. O fígado dói? Pois que doa ainda mais!", 0.95, 0.8],
        ["leitor", "Vivo assim há muito tempo: uns vinte anos. Agora tenho quarenta. Antes eu trabalhava numa repartição; agora não trabalho mais. Fui um funcionário mau. Era grosseiro, e tinha prazer nisso.", 0.95, 0.5],
        ["leitor", "Afinal, eu não aceitava propina, então precisava me recompensar pelo menos com isso.", 0.95, 0.5],
        ["leitor", "(Piada ruim; mas não vou riscá-la. Escrevi achando que ia sair muito espirituosa; e agora que eu mesmo vi que só queria me exibir de um jeito baixo, de propósito não vou riscá-la!)", 0.95, 0.5],
        ["leitor", "Quando os requerentes se aproximavam da minha mesa para pedir informações, eu rangia os dentes para eles, e sentia um prazer implacável quando conseguia magoar alguém. Quase sempre conseguia.", 0.95, 0.5],
        ["leitor", "Na maioria era gente tímida: requerentes, já se sabe. Mas, entre os janotas, havia um oficial que eu não suportava. Ele não queria de jeito nenhum se submeter, e fazia um barulho repugnante com o sabre.", 0.95, 0.5],
        ["leitor", "Por causa desse sabre, ficamos um ano e meio em guerra. No fim, eu venci. Ele parou de fazer barulho. Aliás, isso foi ainda na minha juventude.", 0.95, 0.5],
        ["leitor", "Mas sabem, senhores, qual era o ponto principal da minha raiva? Pois é aí que estava toda a questão, aí é que estava a maior torpeza:", 0.95, 0.5],
        ["leitor", "a todo instante, mesmo no momento de maior bile, eu tinha a consciência vergonhosa de que não só não era um homem mau, como nem sequer era um homem amargurado; de que só espantava os pardais à toa, e com isso me divertia.", 0.95, 0.5],
        ["leitor", "Estou espumando pela boca, mas tragam-me uma bonequinha qualquer, deem-me um chazinho com açúcar, e eu talvez até me acalme. Até me comovo de coração,", 0.95, 0.5],
        ["leitor", "embora depois, com certeza, vá ranger os dentes contra mim mesmo, e passe alguns meses sem dormir, de vergonha. Esse é o meu jeito.", 0.95, 0.8],
        ["leitor", "Isso que eu disse há pouco, que fui um funcionário mau, foi mentira. Menti por raiva. Eu só me divertia, com os requerentes e com o oficial, mas no fundo nunca consegui me tornar mau.", 0.95, 0.5],
        ["leitor", "A todo instante eu sentia em mim muitos e muitos elementos contrários a isso. Sentia que eles fervilhavam dentro de mim, esses elementos contrários.", 0.95, 0.5],
        ["leitor", "Sabia que fervilharam em mim a vida inteira, e que pediam para sair, mas eu não deixava, não deixava, de propósito não deixava que saíssem.", 0.95, 0.5],
        ["leitor", "Eles me torturavam até a vergonha; me levavam a convulsões e, por fim, me cansaram. E como me cansaram!", 0.95, 0.5],
        ["leitor", "Será que não lhes parece, senhores, que agora estou me arrependendo de alguma coisa diante dos senhores, que estou pedindo perdão por alguma coisa?... Tenho certeza de que lhes parece...", 0.95, 0.5],
        ["leitor", "Mas, aliás, garanto que para mim tanto faz, se lhes parece...", 0.95, 1.2],
        ["comentarista", "Repare no método do narrador. Ele abre dizendo: sou um homem mau. Dois parágrafos depois: era mentira, menti por raiva.", 1.0, 0.5],
        ["comentarista", "Ele afirma, desmente e ainda adivinha o que você está pensando dele. Esse vai e vem é o livro inteiro.", 1.0, 0.5],
        ["comentarista", "Uma leitura possível: ele não consegue falar sem imaginar alguém que o julga. Por isso nunca diz nada de uma vez só.", 1.0, 1.2],
        ["leitor", "Eu não só não consegui me tornar mau, como não consegui me tornar coisa nenhuma: nem mau, nem bom, nem canalha, nem honesto, nem herói, nem inseto.", 0.95, 0.5],
        ["leitor", "Agora vou levando a vida no meu canto, provocando a mim mesmo com o consolo raivoso e inútil de que um homem inteligente não pode, a sério, se tornar coisa alguma, e que só o tolo se torna alguma coisa.", 0.95, 0.5],
        ["leitor", "Sim, senhores: o homem inteligente do século dezenove precisa, e tem a obrigação moral, de ser uma criatura essencialmente sem caráter; e o homem de caráter, o homem de ação, uma criatura essencialmente limitada.", 0.95, 0.5],
        ["leitor", "Esta é a minha convicção de quarenta anos. Tenho agora quarenta anos, e quarenta anos são uma vida inteira; é a mais profunda velhice. Viver além dos quarenta é indecente, vulgar, imoral!", 0.95, 0.5],
        ["leitor", "Quem vive mais de quarenta anos? Respondam com sinceridade, com honestidade! Eu lhes digo quem vive: os tolos e os canalhas vivem.", 0.95, 0.5],
        ["leitor", "(Corte: ele repete, aos gritos, que dirá isso na cara de todos os velhos respeitáveis e do mundo inteiro.)", 1.0, 0.6],
        ["leitor", "Eu tenho o direito de falar assim, porque eu mesmo vou viver até os sessenta. Vou viver até os setenta! Vou viver até os oitenta!... Esperem! Deixem-me tomar fôlego...", 0.95, 1.0],
        ["leitor", "Os senhores provavelmente acham que eu quero fazê-los rir? Também nisso se enganaram. Não sou de jeito nenhum um homem tão alegre quanto lhes parece, ou quanto talvez lhes pareça;", 0.95, 0.5],
        ["leitor", "aliás, se os senhores, irritados com toda esta tagarelice (e eu já sinto que estão irritados), resolverem me perguntar: afinal, quem é você? Eu lhes respondo: sou um assessor de colégio.", 0.95, 0.5],
        ["leitor", "Trabalhei para ter o que comer (mas unicamente para isso), e quando, no ano passado, um parente distante me deixou seis mil rublos em testamento, pedi demissão na mesma hora e me instalei no meu canto.", 0.95, 0.5],
        ["leitor", "Eu já morava antes neste canto, mas agora me instalei neste canto. Meu quarto é ruim, horrível, na beira da cidade.", 0.95, 0.5],
        ["leitor", "(Corte: ele descreve a criada e os conselhos para deixar Petersburgo, por causa do clima e do custo de vida.)", 1.0, 0.6],
        ["leitor", "Mas eu fico em Petersburgo; não vou sair de Petersburgo! Não vou sair porque... Ah! Mas isso é absolutamente indiferente: se eu saio ou não saio.", 0.95, 1.0],
        ["leitor", "Pois bem: sobre o que um homem decente pode falar com mais prazer?", 0.95, 1.2],
        ["leitor", "Resposta: sobre si mesmo.", 0.95, 0.8],
        ["leitor", "Pois então eu também vou falar de mim.", 0.95, 1.5],
        ["comentarista", "Um fato para situar o narrador: assessor de colégio era o oitavo dos catorze graus da Tabela de Patentes, a escada do funcionalismo russo criada por Pedro, o Grande. Um posto do meio, sem brilho.", 1.0, 0.5],
        ["comentarista", "O funcionário pequeno é uma tradição da literatura russa. Está em O Capote, de Gógol, e no primeiro romance do próprio Dostoiévski, Gente Pobre, de 1846.", 1.0, 0.5],
        ["comentarista", "A diferença é que o homem do subsolo não pede pena a ninguém. Ele ataca, e ataca primeiro.", 1.0, 0.5],
        ["comentarista", "Um último detalhe: o narrador diz que viver além dos quarenta é indecente. Quando escreveu isso, Dostoiévski tinha quarenta e dois anos.", 1.0, 1.5],
    ]},
    2: {"titulo": "A consciência como doença", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo dois.", 1.0, 0.8],
        ["comentarista", "No capítulo anterior, o narrador disse que não conseguiu ser nada, nem mesmo um inseto. Agora ele vai explicar por quê.", 1.0, 1.0],
        ["leitor", "Capítulo dois.", 0.95, 0.8],
        ["leitor", "Agora quero lhes contar, senhores, queiram ou não queiram ouvir, por que eu nem mesmo um inseto consegui me tornar. Digo-lhes solenemente que muitas vezes quis me tornar um inseto. Mas nem disso fui digno.", 0.95, 0.5],
        ["leitor", "Juro-lhes, senhores, que ter consciência demais é uma doença; uma doença de verdade, completa.", 0.95, 0.5],
        ["leitor", "Para a vida humana comum, bastaria de sobra a consciência humana comum, isto é, a metade, um quarto da porção que cabe ao homem culto do nosso infeliz século dezenove,", 0.95, 0.5],
        ["leitor", "sobretudo se ele tem a dupla infelicidade de morar em Petersburgo, a cidade mais abstrata e mais premeditada de todo o globo terrestre. (Há cidades premeditadas e cidades não premeditadas.)", 0.95, 0.5],
        ["leitor", "Seria perfeitamente suficiente, por exemplo, a consciência com que vivem todas as chamadas pessoas espontâneas e os homens de ação.", 0.95, 0.5],
        ["leitor", "Aposto que os senhores pensam que escrevo tudo isso para me exibir, para fazer graça às custas dos homens de ação, e que, por exibicionismo de mau gosto, faço barulho com o sabre, como o meu oficial.", 0.95, 0.5],
        ["leitor", "Mas, senhores, quem pode se envaidecer das próprias doenças, e ainda se exibir com elas?", 0.95, 0.5],
        ["leitor", "Aliás, o que estou dizendo? Todos fazem isso; é justamente das doenças que as pessoas se envaidecem, e eu talvez mais que todos. Não vamos discutir; a minha objeção é absurda.", 0.95, 0.5],
        ["leitor", "Mas, mesmo assim, estou firmemente convencido de que não só consciência demais, mas qualquer consciência é uma doença. Insisto nisso.", 0.95, 1.2],
        ["comentarista", "Petersburgo foi fundada em 1703, por ordem de Pedro, o Grande, num pântano à beira do mar, para ser a capital moderna da Rússia. Daí a piada: é uma cidade feita de propósito, quase uma ideia de cidade.", 1.0, 0.5],
        ["comentarista", "Antes de Dostoiévski, Púchkin já tinha feito de Petersburgo uma personagem, no poema O Cavaleiro de Bronze, e Gógol, nos seus contos de Petersburgo.", 1.0, 1.2],
        ["leitor", "Deixemos isso por um minuto. Digam-me uma coisa: por que acontecia, como que de propósito, que naqueles mesmos minutos, sim, nos mesmíssimos minutos,", 0.95, 0.5],
        ["leitor", "em que eu era mais capaz de perceber todas as sutilezas de tudo o que é belo e sublime, como se dizia entre nós em outros tempos,", 0.95, 0.5],
        ["leitor", "acontecia de eu já não perceber, e sim fazer coisas tão feias, coisas que... bem, numa palavra, que talvez todos façam, mas que, como de propósito, me vinham justamente quando eu tinha mais consciência de que não devia fazê-las de modo algum?", 0.95, 0.5],
        ["leitor", "Quanto mais consciência eu tinha do bem e de todo esse belo e sublime, mais fundo eu afundava no meu lodo, e mais capaz ficava de me atolar nele por completo.", 0.95, 0.5],
        ["leitor", "Mas o traço principal era que tudo isso parecia não estar em mim por acaso, mas como se tivesse de ser assim. Como se fosse o meu estado mais normal, e de modo algum uma doença ou um defeito,", 0.95, 0.5],
        ["leitor", "de modo que, por fim, até perdi a vontade de lutar contra esse defeito. Acabei quase acreditando (e talvez tenha mesmo acreditado) que esse era, talvez, o meu estado normal.", 0.95, 0.5],
        ["leitor", "Mas antes, no começo, quanto tormento eu suportei nessa luta! Eu não acreditava que isso acontecesse com os outros, e por isso guardei a vida inteira como um segredo. Eu tinha vergonha (talvez ainda tenha);", 0.95, 0.5],
        ["leitor", "chegava ao ponto de sentir um prazerzinho secreto, anormal, mesquinho, ao voltar para o meu canto numa daquelas noites mais sórdidas de Petersburgo,", 0.95, 0.5],
        ["leitor", "e ter a consciência aguda de que hoje, de novo, eu tinha feito uma sujeira, e que o que estava feito não tinha mais volta;", 0.95, 0.5],
        ["leitor", "e, por dentro, em segredo, roer, roer a mim mesmo com os dentes, serrar e chupar a mim mesmo, até que a amargura se transformava, enfim, numa doçura vergonhosa, maldita, e, por fim, num prazer decidido, sério!", 0.95, 0.5],
        ["leitor", "Sim, num prazer, num prazer! Insisto nisso.", 0.95, 1.2],
        ["comentarista", "A expressão o belo e o sublime vem da estética do século dezoito, de Kant e de Schiller, e virou bordão dos idealistas russos dos anos 1840.", 1.0, 0.5],
        ["comentarista", "Dostoiévski foi um deles. Em 1849, foi preso por frequentar um círculo de discussão, o de Petrachévski, passou por uma falsa execução e cumpriu quatro anos de trabalhos forçados na Sibéria.", 1.0, 0.5],
        ["comentarista", "Isso é fato. Agora, uma interpretação: aqui ele parece rir do idealista que ele mesmo foi, o homem que sente o sublime e, no minuto seguinte, faz a coisa feia.", 1.0, 1.2],
        ["leitor", "Comecei a falar justamente porque quero saber ao certo: será que os outros também têm prazeres assim? Vou lhes explicar: o prazer vinha justamente da consciência viva demais da própria humilhação;", 0.95, 0.5],
        ["leitor", "de sentir que já se chegou à última parede; que isso é ruim, mas que não pode ser de outro jeito; que não há saída, que você nunca vai se tornar outra pessoa;", 0.95, 0.5],
        ["leitor", "que, mesmo que ainda houvesse tempo e fé para se transformar em outra coisa, você mesmo, com certeza, não ia querer se transformar;", 0.95, 0.5],
        ["leitor", "e, se quisesse, também não faria nada, porque, na verdade, talvez nem exista em que se transformar.", 0.95, 0.5],
        ["leitor", "(Corte: ele atribui tudo às leis da consciência intensificada, e à inércia que vem delas.)", 1.0, 0.6],
        ["leitor", "Mas chega... Ah, quanta coisa eu amontoei, e o que expliquei?... Como se explica aqui o prazer? Mas eu vou me explicar! Vou levar isso até o fim! Foi para isso mesmo que peguei a pena...", 0.95, 1.0],
        ["leitor", "Eu, por exemplo, sou terrivelmente vaidoso. Sou desconfiado e suscetível como um corcunda ou um anão, mas, palavra, tive momentos em que, se me dessem uma bofetada, talvez eu até ficasse contente.", 0.95, 0.5],
        ["leitor", "Falo sério: com certeza eu saberia encontrar até nisso o meu tipo de prazer; naturalmente, o prazer do desespero. Mas é no desespero que estão os prazeres mais ardentes,", 0.95, 0.5],
        ["leitor", "sobretudo quando se tem uma consciência muito forte de que a situação não tem saída. E ali, com a bofetada, a consciência de como fomos reduzidos a pó esmaga de vez.", 0.95, 0.5],
        ["leitor", "O principal, porém, é que, por mais voltas que se dê, sempre acabo sendo o primeiro culpado de tudo e, o que é mais ofensivo, culpado sem culpa e, por assim dizer, pelas leis da natureza.", 0.95, 0.5],
        ["leitor", "Culpado, em primeiro lugar, porque sou mais inteligente do que todos os que me cercam.", 0.95, 0.5],
        ["leitor", "(Eu sempre me considerei mais inteligente do que todos os que me cercam, e às vezes, acreditem, até me envergonhava disso. Pelo menos, a vida inteira olhei meio de lado, e nunca consegui olhar as pessoas nos olhos.)", 0.95, 0.5],
        ["leitor", "Culpado, enfim, porque, se houvesse em mim alguma generosidade, eu só teria mais tormento com a consciência de toda a sua inutilidade.", 0.95, 0.5],
        ["leitor", "Afinal, eu com certeza não saberia fazer nada com a minha generosidade: nem perdoar, porque o ofensor talvez tenha me batido pelas leis da natureza, e as leis da natureza não se perdoam;", 0.95, 0.5],
        ["leitor", "nem esquecer, porque, ainda que sejam as leis da natureza, ofende do mesmo jeito.", 0.95, 0.5],
        ["leitor", "Por fim, mesmo que eu quisesse ser nada generoso e, ao contrário, desejasse me vingar do ofensor, eu não conseguiria me vingar de ninguém em nada, porque com certeza não me decidiria a fazer coisa alguma, mesmo que pudesse.", 0.95, 0.5],
        ["leitor", "Por que não me decidiria? Sobre isso, quero dizer duas palavras à parte.", 0.95, 1.5],
        ["comentarista", "Guarde duas imagens deste capítulo, porque elas voltam: a última parede, onde ele diz ter chegado, e as leis da natureza, que não dá para perdoar. No capítulo três, a parede vira o centro da discussão.", 1.0, 0.5],
        ["comentarista", "Fora do livro, dois ecos. Dois anos depois, em 1866, Dostoiévski publicou Crime e Castigo, cujo herói, Raskólnikov, também pensa demais antes de agir.", 1.0, 0.5],
        ["comentarista", "E em 1887 Nietzsche encontrou este livro, por acaso, numa tradução francesa, e contou numa carta ao amigo Overbeck que reconheceu ali um parente. É um fato curioso; não prova que um influenciou o outro.", 1.0, 1.5],
    ]},
    3: {"titulo": "A parede", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo três.", 1.0, 0.8],
        ["comentarista", "O capítulo anterior terminou com uma pergunta: por que o narrador não conseguiria se vingar? A resposta traz a imagem mais famosa do livro: a parede.", 1.0, 1.0],
        ["leitor", "Capítulo três.", 0.95, 0.8],
        ["leitor", "Pois as pessoas que sabem se vingar, e em geral se defender, como é que fazem isso? Quando o sentimento de vingança toma conta delas, nada mais resta em todo o seu ser, naquele momento, além desse sentimento.", 0.95, 0.5],
        ["leitor", "Um senhor desses vai direto ao alvo, como um touro enfurecido, de chifres baixos, e só uma parede, talvez, consegue pará-lo.", 0.95, 0.5],
        ["leitor", "(A propósito: diante da parede, esses senhores, isto é, as pessoas espontâneas e os homens de ação, se rendem sinceramente.", 0.95, 0.5],
        ["leitor", "Para eles, a parede não é uma desculpa, como para nós, as pessoas que pensam e, portanto, não fazem nada; não é um pretexto para voltar atrás, pretexto em que a gente como nós em geral nem acredita, mas que sempre recebe com alegria.", 0.95, 0.5],
        ["leitor", "Não, eles se rendem com toda a sinceridade. A parede tem para eles algo de tranquilizador, de moralmente libertador e definitivo, talvez até algo de místico... Mas da parede eu falo depois.)", 0.95, 0.5],
        ["leitor", "Pois bem, é esse homem espontâneo que eu considero o homem verdadeiro, normal, como o quis ver a própria mãe natureza, tão carinhosa, ao fazê-lo nascer na terra. Invejo esse homem até a última gota de bile.", 0.95, 0.5],
        ["leitor", "Ele é tolo, nisso eu não discuto com os senhores; mas talvez o homem normal deva mesmo ser tolo, como é que os senhores sabem? Talvez isso seja até muito bonito.", 0.95, 0.5],
        ["leitor", "E estou ainda mais convencido dessa suspeita, digamos, maldosa, porque, se pegarmos o contrário do homem normal, isto é, o homem de consciência aguda, que saiu, é claro, não do seio da natureza, mas de uma retorta de laboratório", 0.95, 0.5],
        ["leitor", "(isso já é quase misticismo, senhores, mas eu suspeito também disso), esse homem de retorta às vezes se rende tanto diante do seu contrário, que ele mesmo, com toda a sua consciência aguda, se considera honestamente um rato, e não um homem.", 0.95, 0.5],
        ["leitor", "(Corte: ele insiste que é um rato de consciência aguda, mas um rato, e que ninguém lhe pediu para se ver assim.)", 1.0, 0.6],
        ["leitor", "Vejamos agora esse rato em ação. Suponhamos, por exemplo, que ele também foi ofendido (e ele quase sempre está ofendido) e que também quer se vingar.", 0.95, 0.5],
        ["leitor", "Talvez se acumule nele ainda mais raiva do que no homem da natureza e da verdade. O desejozinho sujo e baixo de pagar ao ofensor com o mesmo mal talvez arranhe dentro dele de um jeito ainda mais sujo do que no homem da natureza e da verdade,", 0.95, 0.5],
        ["leitor", "porque o homem da natureza e da verdade, por sua tolice de nascença, considera a sua vingança simplesmente justiça; já o rato, por causa da consciência aguda, nega que ali haja justiça.", 0.95, 0.5],
        ["leitor", "Chega-se finalmente ao ato, ao próprio ato da vingança. O infeliz rato, além da sujeira inicial, já conseguiu amontoar à sua volta, na forma de perguntas e dúvidas, tantas outras sujeiras; ligou a uma pergunta tantas perguntas sem resposta,", 0.95, 0.5],
        ["leitor", "que, sem querer, junta-se ao seu redor uma lavagem fatal, uma lama fedorenta, feita das suas dúvidas, das suas agitações e, por fim, das cusparadas que caem sobre ele vindas dos homens de ação, que o rodeiam solenemente como juízes e ditadores, e riem dele a plenos pulmões.", 0.95, 0.5],
        ["leitor", "É claro que só lhe resta abanar a patinha para tudo e, com um sorriso de falso desprezo, em que nem ele mesmo acredita, enfiar-se vergonhosamente na sua fresta.", 0.95, 0.5],
        ["leitor", "Lá, no seu subsolo nojento e fedorento, o nosso rato ofendido, surrado e ridicularizado mergulha imediatamente numa raiva fria, venenosa e, principalmente, eterna.", 0.95, 0.8],
        ["leitor", "Durante quarenta anos seguidos vai relembrar a ofensa até os últimos detalhes, os mais vergonhosos, e a cada vez vai acrescentar por conta própria detalhes ainda mais vergonhosos, provocando e irritando a si mesmo, com raiva, com a própria fantasia.", 0.95, 0.5],
        ["leitor", "Vai ter vergonha da própria fantasia, mas mesmo assim vai relembrar tudo, revirar tudo, inventar contra si coisas que nunca aconteceram, com o pretexto de que também poderiam ter acontecido, e não vai perdoar nada.", 0.95, 0.5],
        ["leitor", "Talvez até comece a se vingar, mas aos pedaços, com mesquinharias, escondido atrás do fogão, incógnito, sem acreditar nem no seu direito de se vingar, nem no sucesso da vingança,", 0.95, 0.5],
        ["leitor", "e sabendo de antemão que, com todas as suas tentativas, vai sofrer cem vezes mais do que aquele de quem se vinga, e que este talvez nem se coce. No leito de morte vai relembrar tudo de novo, com os juros acumulados de todo esse tempo, e...", 0.95, 0.5],
        ["leitor", "Mas é justamente nesse meio desespero frio e repugnante, nessa meia fé, nesse enterrar-se vivo de propósito, de desgosto, no subsolo por quarenta anos,", 0.95, 0.5],
        ["leitor", "nessa situação sem saída criada com esforço e ainda assim um pouco duvidosa, em todo esse veneno de desejos não satisfeitos que se voltaram para dentro, em toda essa febre de hesitações, de decisões tomadas para sempre e de arrependimentos que voltam um minuto depois,", 0.95, 0.5],
        ["leitor", "é aí que está o suco daquele estranho prazer de que eu falava. Ele é tão sutil, às vezes escapa tanto à consciência, que pessoas um pouquinho limitadas, ou simplesmente pessoas de nervos fortes, não vão entender nele um único traço.", 0.95, 0.8],
        ["leitor", "Talvez também não entendam, os senhores vão acrescentar, rindo, aqueles que nunca levaram uma bofetada. E assim vão insinuar, educadamente, que eu talvez já tenha levado uma bofetada na vida, e por isso falo como entendido. Aposto que os senhores pensam isso.", 0.95, 0.5],
        ["leitor", "Mas fiquem tranquilos, senhores: eu nunca levei bofetadas, embora para mim tanto faz o que os senhores pensem disso. Talvez eu até lamente ter distribuído poucas bofetadas na vida. Mas chega, nem mais uma palavra sobre esse tema tão interessante para os senhores.", 0.95, 1.2],
        ["comentarista", "O homem da natureza e da verdade aparece em francês no original: l'homme de la nature et de la vérité. Os comentadores ligam a expressão a Rousseau, que nas Confissões se propõe mostrar um homem em toda a verdade da natureza.", 1.0, 0.5],
        ["comentarista", "Repare na troca de imagem: o narrador deixou de ser inseto e virou rato. E o rato mora numa fresta, no subsolo. É daqui que vem o título do livro.", 1.0, 1.2],
        ["leitor", "Continuo, com calma, a falar das pessoas de nervos fortes, que não entendem certo refinamento dos prazeres. Esses senhores, em certas ocasiões, berram como touros a plenos pulmões, e isso, digamos, até lhes dá muita honra; mas, como eu já disse, diante do impossível eles logo se resignam.", 0.95, 0.5],
        ["leitor", "O impossível quer dizer a parede de pedra? Que parede de pedra? Ora, é claro: as leis da natureza, as conclusões das ciências naturais, a matemática.", 0.95, 0.5],
        ["leitor", "Quando lhe provarem, por exemplo, que você descende do macaco, não adianta franzir a testa: aceite como é.", 0.95, 0.5],
        ["leitor", "Quando lhe provarem que, no fundo, uma gotinha da sua própria gordura deve lhe ser mais cara do que cem mil semelhantes seus, e que nesse resultado se resolvem, no fim, todas as chamadas virtudes e obrigações e outros delírios e preconceitos,", 0.95, 0.5],
        ["leitor", "aceite assim mesmo, não há o que fazer, porque dois e dois é matemática. Tentem contestar.", 0.95, 0.8],
        ["leitor", "Ora, por favor, vão gritar para os senhores, não se pode revoltar-se: isto é dois e dois são quatro! A natureza não pede a sua opinião; ela não quer saber dos seus desejos, nem se as leis dela lhe agradam ou não.", 0.95, 0.5],
        ["leitor", "Os senhores são obrigados a aceitá-la como ela é, e, portanto, também todos os seus resultados. A parede, então, é a parede... e assim por diante.", 0.95, 0.5],
        ["leitor", "Meu Deus, mas o que me importam as leis da natureza e da aritmética, se por algum motivo essas leis e o dois e dois são quatro não me agradam?", 0.95, 0.5],
        ["leitor", "É claro que não vou derrubar uma parede dessas com a testa, se de fato não tiver forças para derrubá-la; mas também não vou me conformar com ela só porque tenho uma parede de pedra e não tive forças.", 0.95, 0.8],
        ["leitor", "Como se uma parede de pedra dessas fosse mesmo um consolo, e contivesse de fato alguma palavra de paz, unicamente porque ela é dois e dois são quatro. Oh, absurdo dos absurdos!", 0.95, 0.5],
        ["leitor", "Muito melhor é entender tudo, ter consciência de tudo, de todas as impossibilidades e paredes de pedra; não se conformar com nenhuma dessas impossibilidades e paredes de pedra, se lhe dá nojo se conformar;", 0.95, 0.5],
        ["leitor", "chegar, pelas combinações lógicas mais inevitáveis, às conclusões mais repugnantes sobre o tema eterno de que até da parede de pedra você é, de algum modo, culpado, embora, de novo, seja claro como o dia que você não tem culpa nenhuma;", 0.95, 0.5],
        ["leitor", "e, por isso, rangendo os dentes em silêncio e impotente, ficar parado na inércia com volúpia, sonhando que nem ter raiva você pode, porque não há de quem; que o objeto não se encontra, e talvez nunca se encontre;", 0.95, 0.5],
        ["leitor", "que ali há uma troca, uma trapaça, uma batota, que ali é só uma lavagem, não se sabe o quê nem se sabe quem; mas, apesar de todas essas incertezas e trapaças, mesmo assim dói, e quanto mais incerto, mais dói!", 0.95, 1.5],
        ["comentarista", "Para situar a parede: em 1863, um ano antes deste livro, Tchernychévski publicou o romance O que fazer?, que defendia o egoísmo racional: a ideia de que o homem, bem informado pela ciência, sempre escolhe o próprio interesse, e daí nasce o bem comum.", 1.0, 0.5],
        ["comentarista", "A gotinha de gordura mais cara que cem mil semelhantes é uma caricatura dessa ideia. E o macaco lembra Darwin: A Origem das Espécies é de 1859, e a primeira tradução russa saiu justamente em 1864.", 1.0, 0.5],
        ["comentarista", "Um eco de outro livro: em 1984, de George Orwell, o regime obriga Winston a aceitar que dois e dois são cinco. Aqui é o contrário: o dois e dois são quatro é que parece a prisão. Uma leitura possível é que os dois livros defendem a mesma coisa, a liberdade de não ser reduzido a uma conta.", 1.0, 1.5],
    ]},
    4: {"titulo": "A dor de dente", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo quatro.", 1.0, 0.8],
        ["comentarista", "Um capítulo curto, com um exemplo bem concreto do prazer na dor: a dor de dente.", 1.0, 1.0],
        ["leitor", "Capítulo quatro.", 0.95, 0.8],
        ["leitor", "Ha, ha, ha! Então, depois disso, o senhor vai achar prazer até na dor de dente! os senhores vão exclamar, rindo.", 0.95, 0.6],
        ["leitor", "E por que não? Também na dor de dente há prazer, eu respondo. Fiquei com dor de dente um mês inteiro; sei que há. Aí, é claro, não se tem raiva em silêncio: geme-se. Mas não são gemidos francos, são gemidos com malícia, e na malícia é que está tudo.", 0.95, 0.5],
        ["leitor", "É nesses gemidos que se expressa o prazer de quem sofre; se não sentisse prazer neles, nem gemeria. É um bom exemplo, senhores, e vou desenvolvê-lo.", 0.95, 0.5],
        ["leitor", "Nesses gemidos se expressa, em primeiro lugar, toda a falta de sentido da sua dor, humilhante para a nossa consciência; toda a legalidade da natureza, para a qual os senhores, naturalmente, estão pouco ligando, mas pela qual sofrem mesmo assim, e ela não.", 0.95, 0.5],
        ["leitor", "Expressa-se a consciência de que não se encontra um inimigo, mas a dor existe; a consciência de que, com todos os dentistas do mundo, os senhores são completamente escravos dos seus dentes;", 0.95, 0.5],
        ["leitor", "de que, se alguém quiser, os dentes param de doer, e, se não quiser, vão doer mais três meses; e de que, por fim, se os senhores ainda não concordam e protestam, só lhes resta, para consolo próprio, açoitar a si mesmos ou bater com mais força o punho na sua parede, e mais nada.", 0.95, 0.5],
        ["leitor", "Pois bem: é dessas ofensas sangrentas, dessas zombarias de não se sabe quem, que começa enfim o prazer, que às vezes chega à mais alta volúpia.", 0.95, 0.8],
        ["leitor", "Peço-lhes, senhores: escutem um dia os gemidos de um homem culto do século dezenove que sofre dos dentes, lá pelo segundo ou terceiro dia, quando já não geme como gemia no primeiro dia, isto é, não simplesmente porque os dentes doem;", 0.95, 0.5],
        ["leitor", "não como um mujique grosseiro qualquer, mas como geme um homem tocado pelo desenvolvimento e pela civilização europeia, um homem separado do solo e das raízes do povo, como se diz hoje em dia.", 0.95, 0.5],
        ["leitor", "Os gemidos dele ficam ruins, maldosos de um jeito sórdido, e duram dias e noites inteiros. E ele mesmo sabe que não vai ganhar nada gemendo; sabe melhor que ninguém que só está esgotando e irritando a si e aos outros à toa;", 0.95, 0.5],
        ["leitor", "sabe que até o público para quem se esforça, e toda a sua família, já o escutam com nojo, não acreditam nele nem um tostão, e entendem por dentro que ele poderia gemer de outro jeito, mais simples, sem floreios, e que só faz isso por raiva e malícia.", 0.95, 0.5],
        ["leitor", "Pois é em todas essas consciências e vergonhas que está a volúpia. Como quem diz: eu incomodo vocês, despedaço o coração de vocês, não deixo ninguém dormir na casa. Pois então não durmam, sintam vocês também, a cada minuto, que os meus dentes doem.", 0.95, 0.5],
        ["leitor", "Para vocês eu já não sou o herói que antes queria parecer, mas simplesmente um homenzinho sujo, um canalha. Pois que seja! Fico muito contente que vocês tenham me desmascarado. É ruim ouvir os meus gemidozinhos mesquinhos? Pois que seja ruim; agora mesmo vou fazer um floreio ainda pior...", 0.95, 0.8],
        ["leitor", "Nem agora entendem, senhores? Não, pelo jeito é preciso desenvolver-se e tomar consciência muito a fundo para entender todas as curvas dessa volúpia! Os senhores riem? Fico muito contente.", 0.95, 0.5],
        ["leitor", "As minhas piadas, senhores, são de mau gosto, é claro, irregulares, confusas, cheias de desconfiança de mim mesmo. Mas é porque eu não respeito a mim mesmo. Será que um homem consciente pode respeitar a si mesmo, por pouco que seja?", 0.95, 1.5],
        ["comentarista", "No original, o narrador fala de todos os Wagenheim do mundo. Wagenheim, segundo os comentadores, era o nome de um dentista de Petersburgo, cujas placas estavam por toda a cidade. Por isso traduzimos como todos os dentistas do mundo.", 1.0, 0.5],
        ["comentarista", "E a expressão separado do solo e das raízes do povo era vocabulário do próprio grupo de Dostoiévski. As revistas dos irmãos, O Tempo e a Época, defendiam a volta ao solo russo. Aqui o narrador usa o bordão com ironia, e o alvo é o próprio campo do autor.", 1.0, 1.5],
    ]},
    5: {"titulo": "O tédio e a inércia", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo cinco.", 1.0, 0.8],
        ["comentarista", "Aqui o narrador explica por que quem pensa demais não age. É o capítulo mais filosófico até agora.", 1.0, 1.0],
        ["leitor", "Capítulo cinco.", 0.95, 0.8],
        ["leitor", "Será possível, será possível que um homem respeite a si mesmo, por pouco que seja, se até no próprio sentimento de humilhação ele tentou encontrar prazer? Não digo isso agora por algum arrependimento açucarado.", 0.95, 0.5],
        ["leitor", "E, em geral, eu nunca suportei dizer: Desculpe, papai, não faço mais. Não porque fosse incapaz de dizer isso, mas, ao contrário, talvez justamente porque era capaz demais, e como!", 0.95, 0.5],
        ["leitor", "Como de propósito, eu me metia em encrenca justamente quando não tinha culpa nenhuma, nem em sonho. Isso era o mais sórdido de tudo. E nessas horas eu de novo me comovia de alma, me arrependia, derramava lágrimas e, é claro, enganava a mim mesmo, embora não estivesse fingindo de jeito nenhum.", 0.95, 0.5],
        ["leitor", "(Corte: ele diz que nem podia culpar as leis da natureza por isso, embora elas o ofendessem a vida inteira.)", 1.0, 0.6],
        ["leitor", "É sórdido lembrar tudo isso, e já era sórdido naquela época. Pois, um minuto depois, eu já percebia com raiva que tudo aquilo era mentira, mentira, uma mentira repugnante e fingida: todos esses arrependimentos, todas essas comoções, todas essas promessas de renascimento.", 0.95, 0.5],
        ["leitor", "E, se me perguntarem para que eu me retorcia e me torturava assim, a resposta é: porque era muito tedioso ficar de braços cruzados; então eu partia para as invenções. Palavra, é assim. Observem melhor a si mesmos, senhores, e vão entender que é assim.", 0.95, 0.5],
        ["leitor", "Eu mesmo inventava aventuras para mim e compunha uma vida, só para viver de algum jeito. Quantas vezes me aconteceu, por exemplo, de me ofender assim, sem motivo, de propósito;", 0.95, 0.5],
        ["leitor", "e a gente mesmo sabe que se ofendeu sem motivo, que fingiu, mas vai se levando a tal ponto que, no fim, palavra, se ofende de verdade. A vida inteira tive vontade de aprontar essas coisas, a ponto de, no fim, já não ter domínio sobre mim mesmo.", 0.95, 0.5],
        ["leitor", "Outra vez quis me apaixonar à força, e foram até duas vezes. E eu sofria, senhores, garanto. No fundo da alma não se acredita que se sofre, a zombaria se mexe, mas mesmo assim eu sofria, e de um jeito verdadeiro, legítimo; sentia ciúme, perdia a cabeça...", 0.95, 0.5],
        ["leitor", "E tudo por tédio, senhores, tudo por tédio; a inércia me esmagou. Pois o fruto direto, legítimo, imediato da consciência é a inércia, isto é, ficar de braços cruzados de propósito. Já mencionei isso acima.", 0.95, 0.8],
        ["leitor", "Repito, repito com força: todas as pessoas espontâneas e os homens de ação são ativos justamente porque são obtusos e limitados. Como explicar isso? Assim:", 0.95, 0.5],
        ["leitor", "por causa da sua limitação, eles tomam as causas mais próximas e secundárias pelas primeiras, e assim se convencem mais depressa e mais fácil que os outros de que encontraram um fundamento indiscutível para o que fazem, e ficam tranquilos; e isso é o principal.", 0.95, 0.5],
        ["leitor", "Pois, para começar a agir, é preciso antes estar completamente tranquilo, sem nenhuma dúvida. Mas como é que eu, por exemplo, vou me tranquilizar? Onde estão as minhas causas primeiras, em que vou me apoiar, onde estão os fundamentos? De onde vou tirá-los?", 0.95, 0.5],
        ["leitor", "Eu me exercito no pensamento, e, portanto, toda causa primeira logo arrasta consigo outra, ainda mais primeira, e assim por diante, até o infinito. Esta é justamente a essência de toda consciência e de todo pensamento. Isso, então, de novo são as leis da natureza.", 0.95, 0.8],
        ["leitor", "E qual é, enfim, o resultado? O mesmo de sempre. Lembrem: há pouco eu falava da vingança. (Os senhores, com certeza, não prestaram atenção.) Foi dito: o homem se vinga porque encontra nisso justiça.", 0.95, 0.5],
        ["leitor", "Quer dizer que ele encontrou a causa primeira, encontrou o fundamento: a justiça. Então ele está tranquilo de todos os lados, e, portanto, se vinga com calma e com sucesso, convencido de que faz uma coisa honesta e justa.", 0.95, 0.5],
        ["leitor", "Mas eu não vejo justiça nisso, nem encontro nenhuma virtude, e, portanto, se eu for me vingar, será só por raiva. A raiva, é claro, poderia vencer tudo, todas as minhas dúvidas, e poderia muito bem servir no lugar da causa primeira, justamente porque ela não é uma causa.", 0.95, 0.5],
        ["leitor", "Mas o que fazer, se eu nem raiva tenho? (Foi por aí mesmo que eu comecei.) A minha raiva, de novo por causa dessas malditas leis da consciência, sofre uma decomposição química.", 0.95, 0.5],
        ["leitor", "Quando se vê, o objeto se evapora, as razões se volatilizam, o culpado não se encontra, a ofensa vira não uma ofensa, mas um destino, algo como uma dor de dente, de que ninguém tem culpa; e, portanto, de novo só resta a mesma saída: bater com mais força na parede.", 0.95, 0.5],
        ["leitor", "E a gente desiste, porque não encontrou a causa primeira. Experimente se deixar levar pelo sentimento às cegas, sem raciocinar, sem causa primeira, afastando a consciência ao menos por um tempo; odeie ou ame, só para não ficar de braços cruzados.", 0.95, 0.5],
        ["leitor", "Depois de amanhã, no mais tardar, você vai começar a desprezar a si mesmo por ter enganado a si mesmo, sabendo o que fazia. Resultado: bolha de sabão e inércia.", 0.95, 0.8],
        ["leitor", "Oh, senhores, talvez eu me considere um homem inteligente só porque a vida inteira não consegui nem começar nem terminar nada. Que seja, que eu seja um tagarela, um tagarela inofensivo e aborrecido, como todos nós.", 0.95, 0.5],
        ["leitor", "Mas o que fazer, se a vocação direta e única de todo homem inteligente é a tagarelice, isto é, despejar de propósito do vazio para o oco?", 0.95, 1.5],
        ["comentarista", "Este capítulo conversa com uma peça que Dostoiévski conhecia bem: Hamlet, de Shakespeare. No monólogo do ser ou não ser, Hamlet diz que a consciência faz de todos nós covardes, e que a reflexão tira das grandes empresas o nome de ação.", 1.0, 0.5],
        ["comentarista", "A diferença, numa leitura possível: Hamlet sofre por não agir e ainda quer agir. O homem do subsolo transformou a inação em teoria, e quase em orgulho.", 1.0, 1.5],
    ]},
    6: {"titulo": "O preguiçoso", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo seis.", 1.0, 0.8],
        ["comentarista", "O mais curto e o mais engraçado da primeira parte. O narrador sonha com uma qualidade bem simples, que ele não tem: ser preguiçoso.", 1.0, 1.0],
        ["leitor", "Capítulo seis.", 0.95, 0.8],
        ["leitor", "Ah, se eu não fizesse nada só por preguiça! Meu Deus, como eu me respeitaria então. Respeitaria justamente porque pelo menos preguiça eu seria capaz de ter; ao menos uma qualidade em mim seria positiva, uma de que eu mesmo teria certeza.", 0.95, 0.5],
        ["leitor", "Pergunta: quem é ele? Resposta: um preguiçoso. Mas isso seria agradabilíssimo de ouvir a meu respeito. Quer dizer que estou definido de modo positivo, quer dizer que há algo a dizer sobre mim.", 0.95, 0.5],
        ["leitor", "Preguiçoso! Mas isso é um título e uma vocação, é uma carreira. Não brinquem, é assim. Eu seria então, por direito, membro do clube mais importante, e só me ocuparia de me respeitar sem parar.", 0.95, 0.5],
        ["leitor", "Conheci um senhor que a vida inteira se orgulhou de entender de vinho Lafite. Ele considerava isso uma virtude positiva e nunca duvidou de si. Morreu não só com a consciência tranquila, mas triunfante, e tinha toda a razão.", 0.95, 0.8],
        ["leitor", "E eu então escolheria para mim uma carreira: seria preguiçoso e comilão, mas não um qualquer, e sim, por exemplo, um que se comove com tudo o que é belo e sublime. Que tal? Faz tempo que sonho com isso.", 0.95, 0.5],
        ["leitor", "Esse belo e sublime pesou bastante na minha nuca nos meus quarenta anos; mas isso nos meus quarenta anos, e naquela época, oh, naquela época seria diferente!", 0.95, 0.5],
        ["leitor", "Eu logo encontraria uma atividade adequada: beber à saúde de tudo o que é belo e sublime. Aproveitaria qualquer ocasião para primeiro derramar uma lágrima no meu cálice, e depois bebê-lo por tudo o que é belo e sublime.", 0.95, 0.5],
        ["leitor", "Eu transformaria tudo no mundo em belo e sublime; na porcaria mais sórdida e indiscutível eu encontraria o belo e sublime. Ficaria lacrimoso como uma esponja molhada.", 0.95, 0.5],
        ["leitor", "Um pintor, por exemplo, pintou um quadro de Gue. Imediatamente bebo à saúde do pintor que pintou o quadro de Gue, porque amo tudo o que é belo e sublime. Um autor escreveu Como quiserem; imediatamente bebo à saúde de quem quiserem, porque amo tudo o que é belo e sublime.", 0.95, 0.5],
        ["leitor", "Por isso exigirei respeito, e perseguirei quem não me respeitar. Vivo tranquilo, morro solenemente; mas isso é uma delícia, uma delícia completa!", 0.95, 0.5],
        ["leitor", "E eu deixaria crescer uma barriga tal, construiria um queixo triplo tal, desenvolveria um nariz cor de vinho tal, que qualquer um que me encontrasse diria, olhando para mim: Isso é que é um mais! Isso é que é positivo de verdade!", 0.95, 0.5],
        ["leitor", "E, digam o que quiserem, senhores, é agradabilíssimo ouvir essas opiniões no nosso século negativo.", 0.95, 1.5],
        ["comentarista", "Duas referências da época, segundo os comentadores. O quadro de Gue é A Última Ceia, de Nikolai Gue, exposto em Petersburgo em 1863, que causou polêmica. E Como quiserem é um conto de Saltykov-Chtchedrin, publicado no mesmo ano na revista O Contemporâneo, dos radicais.", 1.0, 0.5],
        ["comentarista", "Repare que o belo e sublime do capítulo dois voltou, agora como piada de mesa de bar. E repare no positivo e no negativo: o narrador chama o seu tempo de século negativo. Ele quer ser alguma coisa, qualquer coisa, nem que seja um gordo satisfeito.", 1.0, 1.5],
    ]},
    7: {"titulo": "O palácio de cristal", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo sete.", 1.0, 0.8],
        ["comentarista", "Aqui começa o centro da Parte um: a discussão com a ideia de que a razão e a ciência vão consertar o ser humano.", 1.0, 1.0],
        ["leitor", "Capítulo sete.", 0.95, 0.8],
        ["leitor", "Mas tudo isso são sonhos dourados. Oh, digam-me: quem foi o primeiro a anunciar, o primeiro a proclamar que o homem só faz sujeiras porque não conhece os seus verdadeiros interesses;", 0.95, 0.5],
        ["leitor", "e que, se o esclarecessem, se lhe abrissem os olhos para os seus interesses verdadeiros e normais, o homem pararia na mesma hora de fazer sujeiras, e ficaria na mesma hora bom e nobre,", 0.95, 0.5],
        ["leitor", "porque, sendo esclarecido e entendendo as suas verdadeiras vantagens, veria justamente no bem a sua própria vantagem; e, como se sabe, nenhum homem pode agir, sabendo o que faz, contra as próprias vantagens; portanto, faria o bem, por assim dizer, por necessidade?", 0.95, 0.5],
        ["leitor", "Oh, criancinha! Oh, criança pura e inocente! Mas, em primeiro lugar, quando foi que aconteceu, em todos esses milênios, que o homem agisse só pela própria vantagem?", 0.95, 0.8],
        ["leitor", "O que fazer com os milhões de fatos que mostram como as pessoas, sabendo o que faziam, isto é, entendendo perfeitamente as suas verdadeiras vantagens, deixaram-nas em segundo plano e se lançaram por outro caminho, ao risco, à sorte,", 0.95, 0.5],
        ["leitor", "sem que ninguém nem nada as obrigasse, como se justamente não quisessem o caminho indicado, e abriram com teimosia, por capricho, outro caminho, difícil, absurdo, procurando-o quase no escuro?", 0.95, 0.5],
        ["leitor", "Quer dizer que essa teimosia e esse capricho eram mesmo mais agradáveis para elas do que qualquer vantagem... Vantagem! O que é vantagem? E os senhores se atrevem a definir com toda a exatidão em que consiste a vantagem humana?", 0.95, 0.5],
        ["leitor", "E se acontecer que a vantagem humana, às vezes, não só possa, mas até deva consistir justamente em desejar para si, em certos casos, o que é ruim, e não o vantajoso? E, se for assim, se esse caso puder existir, a regra toda vira pó.", 0.95, 0.8],
        ["leitor", "Os senhores riem; riam, senhores, mas respondam: as vantagens humanas foram contadas com total exatidão? Não haverá algumas que não só não couberam, como não podem caber em classificação nenhuma?", 0.95, 0.5],
        ["leitor", "Afinal, os senhores, pelo que sei, tiraram todo o seu registro das vantagens humanas de médias, de números estatísticos e de fórmulas científico-econômicas. As suas vantagens são a prosperidade, a riqueza, a liberdade, a tranquilidade, e assim por diante;", 0.95, 0.5],
        ["leitor", "de modo que um homem que fosse, de modo claro e consciente, contra todo esse registro seria, na opinião dos senhores, e na minha também, é claro, um obscurantista ou um louco completo, não é?", 0.95, 0.5],
        ["leitor", "Mas o espantoso é isto: por que todos esses estatísticos, sábios e amigos do gênero humano, ao calcular as vantagens humanas, sempre deixam uma de fora? E toda a conta depende dela.", 0.95, 0.5],
        ["leitor", "Não seria grande coisa pegar essa vantagem e pôr na lista. Mas a desgraça é justamente que essa vantagem complicada não entra em classificação nenhuma, não cabe em lista nenhuma.", 0.95, 0.8],
        ["leitor", "(Corte: ele descreve um amigo, que é todos nós, que explica com eloquência o que a razão manda fazer e, quinze minutos depois, faz o contrário.)", 1.0, 0.6],
        ["leitor", "Pois então, senhores, não existirá de fato alguma coisa que é mais cara para quase todo homem do que as suas melhores vantagens? Ou, para não violar a lógica, uma vantagem mais vantajosa de todas, justamente essa que fica de fora,", 0.95, 0.5],
        ["leitor", "mais importante e mais vantajosa que todas as outras, pela qual o homem, se for preciso, está pronto a ir contra todas as leis, isto é, contra a razão, a honra, a tranquilidade, a prosperidade,", 0.95, 0.5],
        ["leitor", "contra todas essas coisas belas e úteis, só para alcançar essa vantagem primeira, a mais vantajosa, que lhe é mais cara que tudo?", 0.95, 0.8],
        ["leitor", "Então, é vantagem do mesmo jeito, os senhores me interrompem. Permitam: ainda vamos nos explicar. O que importa não é o trocadilho, mas que essa vantagem é notável justamente porque destrói todas as nossas classificações,", 0.95, 0.5],
        ["leitor", "e despedaça sempre todos os sistemas montados pelos amigos do gênero humano para a felicidade do gênero humano. Numa palavra, atrapalha tudo.", 0.95, 0.8],
        ["leitor", "Mas, antes de nomear essa vantagem, quero me comprometer pessoalmente, e por isso declaro com ousadia que todos esses belos sistemas, todas essas teorias que explicam à humanidade os seus interesses normais,", 0.95, 0.5],
        ["leitor", "para que ela, buscando-os por necessidade, fique logo boa e nobre, por enquanto, na minha opinião, são pura logística! Sim, senhor, logística!", 0.95, 0.5],
        ["leitor", "Afirmar essa teoria da renovação de todo o gênero humano por meio do sistema das suas próprias vantagens é, para mim, quase o mesmo que afirmar, seguindo Buckle, que a civilização torna o homem mais brando e, portanto, menos sanguinário e menos capaz de guerra.", 0.95, 0.5],
        ["leitor", "Pela lógica, parece que dá isso mesmo. Mas o homem é tão apegado ao sistema e à conclusão abstrata, que está pronto a deformar a verdade de propósito, a não ver o que vê e não ouvir o que ouve, só para justificar a sua lógica.", 0.95, 0.5],
        ["leitor", "Olhem em volta: o sangue corre como um rio, e ainda de um jeito tão alegre, como champanhe. Aí está todo o nosso século dezenove, em que também viveu Buckle.", 0.95, 0.5],
        ["leitor", "Aí está Napoleão, o grande e o de agora. Aí está a América do Norte, a união eterna. Aí está, enfim, a caricatura do Schleswig-Holstein... E o que a civilização abranda em nós?", 0.95, 0.5],
        ["leitor", "A civilização só desenvolve no homem a variedade das sensações, e mais nada. E, com o desenvolvimento dessa variedade, o homem talvez chegue a encontrar prazer no sangue. Isso, aliás, já lhe aconteceu.", 0.95, 0.5],
        ["leitor", "Os senhores já repararam que os derramadores de sangue mais refinados foram quase todos senhores civilizadíssimos, a quem os Átilas e os Stienka Rázin às vezes não chegavam aos pés?", 0.95, 0.5],
        ["leitor", "E, se eles não saltam tanto aos olhos quanto Átila e Stienka Rázin, é justamente porque são comuns demais, já nos acostumamos com eles.", 0.95, 0.5],
        ["leitor", "Pelo menos, com a civilização o homem ficou, se não mais sanguinário, com certeza sanguinário de um jeito pior, mais sujo, do que antes. Antes, via justiça no derramamento de sangue, e exterminava quem devia com a consciência tranquila;", 0.95, 0.5],
        ["leitor", "agora, embora consideremos o derramamento de sangue uma sujeira, mesmo assim nos ocupamos dessa sujeira, e mais do que antes. O que é pior? Decidam os senhores.", 0.95, 0.8],
        ["leitor", "(Corte: o exemplo de Cleópatra, que espetava alfinetes de ouro nas escravas, e a resposta de que hoje também se espetam alfinetes.)", 1.0, 0.6],
        ["leitor", "Mas os senhores estão completamente seguros de que o homem vai se acostumar a agir como a razão manda, quando passarem certos velhos maus hábitos, e o bom senso e a ciência tiverem reeducado por completo a natureza humana.", 0.95, 0.5],
        ["leitor", "E mais: então, dizem os senhores, a própria ciência vai ensinar ao homem que, na verdade, ele não tem vontade nem capricho, e nunca teve, e que ele mesmo não passa de algo como uma tecla de piano, ou um pino de realejo;", 0.95, 0.5],
        ["leitor", "e que, além disso, existem no mundo as leis da natureza; de modo que tudo o que ele faz não é feito pela vontade dele, mas sozinho, pelas leis da natureza.", 0.95, 0.5],
        ["leitor", "Portanto, basta descobrir essas leis da natureza, e o homem não vai mais responder pelos seus atos, e viver vai ficar extremamente fácil.", 0.95, 0.5],
        ["leitor", "Todos os atos humanos serão então calculados por essas leis, matematicamente, como uma tábua de logaritmos, até cento e oito mil, e registrados num calendário;", 0.95, 0.5],
        ["leitor", "ou, melhor ainda, vão aparecer publicações bem-intencionadas, como os dicionários enciclopédicos de hoje, em que tudo estará tão exatamente calculado e indicado, que não haverá mais no mundo nem atos nem aventuras.", 0.95, 0.8],
        ["leitor", "Então, são os senhores que dizem, surgirão novas relações econômicas, prontas e também calculadas com precisão matemática, de modo que num instante vão desaparecer todas as perguntas possíveis, justamente porque haverá para elas todas as respostas possíveis.", 0.95, 0.5],
        ["leitor", "Então se erguerá o palácio de cristal. Então... Bem, numa palavra, então chegará voando o pássaro da felicidade.", 0.95, 0.5],
        ["leitor", "É claro que não se pode garantir de jeito nenhum, e isto já sou eu quem diz, que então não vá ser, por exemplo, terrivelmente tedioso, porque o que fazer quando tudo estiver calculado numa tabelinha; em compensação, tudo será extremamente sensato.", 0.95, 0.5],
        ["leitor", "É claro que, por tédio, o que não se inventa! Também os alfinetes de ouro são espetados por tédio. O ruim, e de novo sou eu quem diz, é que, quem sabe, então até se fique contente com os alfinetes de ouro.", 0.95, 0.5],
        ["leitor", "Pois o homem é tolo, fenomenalmente tolo. Quer dizer, ele não é nada tolo, mas em compensação é tão ingrato, que procurando outro igual não se acha.", 0.95, 0.8],
        ["leitor", "Eu, por exemplo, não vou me espantar nem um pouco se, de repente, sem mais nem menos, no meio da sensatez geral do futuro, surgir um cavalheiro com uma cara pouco nobre, ou melhor, retrógrada e zombeteira,", 0.95, 0.5],
        ["leitor", "puser as mãos na cintura e disser a todos nós: e então, senhores, que tal se derrubássemos toda essa sensatez de uma vez, com um pontapé, só para que todos esses logaritmos fossem para o diabo, e pudéssemos viver de novo pela nossa vontade tola!", 0.95, 0.5],
        ["leitor", "Isso ainda não seria nada; o que ofende é que ele vai, com certeza, encontrar seguidores: o homem é feito assim. E tudo pela razão mais vazia, que parece nem valer a pena mencionar:", 0.95, 0.5],
        ["leitor", "é que o homem, sempre e em toda parte, quem quer que fosse, gostou de agir como queria, e não como lhe mandavam a razão e a vantagem; e pode-se querer contra a própria vantagem, e às vezes até se deve, positivamente. Esta já é a minha ideia.", 0.95, 0.8],
        ["leitor", "O próprio querer, livre e solto, o próprio capricho, ainda que o mais selvagem, a própria fantasia, irritada às vezes até a loucura: é isso a tal vantagem esquecida, a mais vantajosa, que não cabe em classificação nenhuma, e por causa da qual todos os sistemas e teorias vão sempre para o diabo.", 0.95, 0.5],
        ["leitor", "E de onde esses sábios tiraram que o homem precisa de um querer normal, virtuoso? De onde imaginaram que o homem precisa de um querer sensatamente vantajoso? O homem só precisa de um querer independente, custe o que custar essa independência, e leve aonde levar.", 0.95, 0.5],
        ["leitor", "E o querer, o diabo sabe...", 0.95, 1.5],
        ["comentarista", "Três fatos para situar o capítulo. Henry Thomas Buckle foi um historiador inglês, autor de uma História da Civilização na Inglaterra, muito lida na Rússia dos anos 1860, que via na civilização um progresso da razão.", 1.0, 0.5],
        ["comentarista", "Os exemplos do narrador são todos da época: Napoleão primeiro e Napoleão terceiro; a Guerra Civil americana, de 1861 a 1865; e a guerra pelo Schleswig-Holstein, travada justamente em 1864, enquanto o livro saía.", 1.0, 0.5],
        ["comentarista", "E o palácio de cristal existia de verdade. Foi o prédio de ferro e vidro da Exposição Universal de Londres, de 1851. Dostoiévski o viu em 1862 e escreveu sobre ele. Em O que fazer?, de Tchernychévski, um palácio assim aparece num sonho da heroína como a imagem da sociedade futura.", 1.0, 0.5],
        ["comentarista", "Uma leitura possível: para o narrador, o palácio de cristal é um mundo sem pergunta nenhuma. E é justamente por isso que ele o teme.", 1.0, 1.5],
    ]},
    8: {"titulo": "A tecla de piano", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo oito.", 1.0, 0.8],
        ["comentarista", "O narrador continua a discussão. A pergunta agora é: e se a ciência provar que a vontade livre não existe?", 1.0, 1.0],
        ["leitor", "Capítulo oito.", 0.95, 0.8],
        ["leitor", "Ha, ha, ha! Mas o querer, no fundo, se o senhor quer saber, nem existe! os senhores me interrompem, gargalhando. A ciência até hoje já conseguiu dissecar tanto o homem, que já sabemos que o querer e a chamada vontade livre não são outra coisa senão...", 0.95, 0.5],
        ["leitor", "Esperem, senhores, eu mesmo queria começar assim. Confesso que até me assustei. Eu ia gritar que o querer depende o diabo sabe de quê, e que isso talvez seja uma graça de Deus, mas me lembrei da ciência e... murchei. E aí os senhores começaram a falar.", 0.95, 0.5],
        ["leitor", "Pois, de fato, se algum dia encontrarem a fórmula de todos os nossos quereres e caprichos, isto é, de que eles dependem, por quais leis acontecem, como se espalham, para onde tendem neste ou naquele caso, uma verdadeira fórmula matemática,", 0.95, 0.5],
        ["leitor", "então o homem talvez pare na mesma hora de querer, e com certeza vai parar. Que graça tem querer por uma tabelinha? E mais: na mesma hora ele vai se transformar de homem num pino de realejo, ou coisa parecida;", 0.95, 0.5],
        ["leitor", "porque o que é um homem sem desejos, sem vontade e sem quereres, senão um pino no cilindro de um realejo? O que acham? Vamos calcular as probabilidades: isso pode acontecer ou não?", 0.95, 0.8],
        ["leitor", "Hum, os senhores decidem. Os nossos quereres, na maior parte, são errados por causa de uma visão errada das nossas vantagens. Às vezes queremos puro disparate justamente porque, por tolice, vemos nesse disparate o caminho mais fácil para alguma vantagem prevista.", 0.95, 0.5],
        ["leitor", "Mas, quando tudo isso estiver explicado, calculado no papel, não haverá mais os chamados desejos. Se um dia o querer combinar por completo com a razão, então vamos raciocinar, e não querer, porque não se pode, conservando a razão, querer um absurdo, e ir contra a razão sabendo o que se faz.", 0.95, 0.5],
        ["leitor", "(Corte: os interlocutores insistem que, descobertas as leis da vontade, cada gesto, até uma figa, estará previsto numa tabela, e que será preciso aceitar até a retorta.)", 1.0, 0.6],
        ["leitor", "Sim, senhor, mas é aí que, para mim, está o problema! Senhores, desculpem-me por filosofar tanto: são quarenta anos de subsolo! Permitam-me fantasiar um pouco.", 0.95, 0.5],
        ["leitor", "Vejam: a razão, senhores, é uma coisa boa, isso é indiscutível; mas a razão é só razão, e satisfaz só a capacidade racional do homem, enquanto o querer é a manifestação da vida inteira, isto é, de toda a vida humana, com a razão e com todas as coçadas.", 0.95, 0.5],
        ["leitor", "E, embora a nossa vida, nessa manifestação, muitas vezes saia uma porcariazinha, mesmo assim é vida, e não só a extração de uma raiz quadrada.", 0.95, 0.5],
        ["leitor", "Eu, por exemplo, quero viver, naturalmente, para satisfazer toda a minha capacidade de viver, e não só a minha capacidade racional, isto é, alguma vigésima parte da minha capacidade de viver.", 0.95, 0.5],
        ["leitor", "O que a razão sabe? A razão só sabe o que conseguiu aprender; e a natureza humana age inteira, com tudo o que há nela, consciente e inconscientemente, e, embora minta, vive.", 0.95, 0.8],
        ["leitor", "Desconfio, senhores, que me olham com pena; repetem que um homem esclarecido e desenvolvido, numa palavra, como será o homem do futuro, não pode querer, sabendo o que faz, nada que lhe seja desvantajoso; que isso é matemática.", 0.95, 0.5],
        ["leitor", "Concordo completamente, é matemática. Mas repito pela centésima vez: há um caso, só um, em que o homem pode, de propósito, conscientemente, desejar para si até o nocivo, o tolo, até o mais tolo:", 0.95, 0.5],
        ["leitor", "é para ter o direito de desejar para si até o mais tolo, e não ficar preso à obrigação de desejar só o inteligente.", 0.95, 0.5],
        ["leitor", "Porque esse mais tolo, esse capricho próprio, pode ser, de fato, senhores, a coisa mais vantajosa de todas que há na terra para gente como nós, mesmo quando nos traz dano evidente e contradiz as conclusões mais sensatas da nossa razão;", 0.95, 0.5],
        ["leitor", "porque, em todo caso, ele nos conserva o mais importante e o mais caro, isto é, a nossa pessoa e a nossa individualidade.", 0.95, 0.8],
        ["leitor", "(Corte: ele admite que o querer às vezes combina com a razão, e diz que discordar dela também pode ser útil, e até louvável.)", 1.0, 0.6],
        ["leitor", "Senhores, suponhamos que o homem não seja tolo. (De fato, não se pode dizer isso dele, nem que seja só porque, se ele for tolo, quem então será inteligente?)", 0.95, 0.5],
        ["leitor", "Mas, se não é tolo, é monstruosamente ingrato! Fenomenalmente ingrato. Acho até que a melhor definição do homem é esta: criatura de duas pernas, e ingrata.", 0.95, 0.5],
        ["leitor", "Mas isso não é tudo; não é o seu defeito principal. O defeito principal é a falta constante de bons modos, constante, desde o Dilúvio Universal até o período Schleswig-Holstein dos destinos humanos. Falta de bons modos e, portanto, falta de sensatez.", 0.95, 0.5],
        ["leitor", "Experimentem lançar um olhar sobre a história da humanidade; o que vão ver? Grandiosa? Talvez, grandiosa. Variada? Talvez, variada. Monótona? Bem, talvez, monótona: brigam e brigam, brigam agora, brigavam antes e vão brigar depois; convenham que é monótono até demais.", 0.95, 0.5],
        ["leitor", "(Corte: piadas sobre o Colosso de Rodes, sobre os uniformes de gala de todos os povos e sobre os sábios virtuosos que, no fim da vida, acabam fazendo algum escândalo.)", 1.0, 0.6],
        ["leitor", "Numa palavra, pode-se dizer tudo da história universal, tudo o que vier à cabeça da imaginação mais transtornada. Só uma coisa não se pode dizer: que é sensata. Engasga-se na primeira palavra.", 0.95, 0.8],
        ["leitor", "Agora eu pergunto: o que se pode esperar do homem, uma criatura dotada de qualidades tão estranhas? Cubram-no de todos os bens da terra, afoguem-no de felicidade até a cabeça, de modo que só subam bolhinhas à superfície da felicidade, como na água;", 0.95, 0.5],
        ["leitor", "deem-lhe tamanha fartura econômica, que não lhe reste mais nada a fazer senão dormir, comer biscoitos de mel e cuidar para que a história universal não acabe; e ainda assim o homem, mesmo aí, só por ingratidão, só por zombaria, vai fazer uma sujeira.", 0.95, 0.5],
        ["leitor", "Vai arriscar até os biscoitos e desejar de propósito o disparate mais destrutivo, o absurdo mais antieconômico, só para misturar a toda essa sensatez positiva o seu elemento fantástico e destrutivo.", 0.95, 0.5],
        ["leitor", "Vai querer conservar justamente os seus sonhos fantásticos, a sua tolice mais vulgar, só para confirmar a si mesmo, como se isso fosse tão necessário, que os homens ainda são homens, e não teclas de piano.", 0.95, 0.8],
        ["leitor", "E mais: mesmo que ele de fato fosse uma tecla de piano, e lhe provassem isso pelas ciências naturais e pela matemática, nem assim ele criaria juízo, mas faria de propósito alguma coisa ao contrário, só por ingratidão, para impor a sua vontade.", 0.95, 0.5],
        ["leitor", "E, se não tiver meios, vai inventar a destruição e o caos, vai inventar sofrimentos variados, e vai impor a sua vontade! Vai lançar uma maldição pelo mundo, e, como só o homem pode amaldiçoar, este é o seu privilégio, o que mais o distingue dos outros animais,", 0.95, 0.5],
        ["leitor", "talvez ele consiga o que quer só com a maldição, isto é, vai se convencer de verdade de que é um homem, e não uma tecla de piano!", 0.95, 0.5],
        ["leitor", "Se os senhores disserem que também tudo isso se pode calcular numa tabelinha, o caos, a treva e a maldição, de modo que a simples possibilidade do cálculo prévio vai parar tudo e a razão vai vencer, então o homem vai ficar louco de propósito, para não ter razão e impor a sua vontade!", 0.95, 0.5],
        ["leitor", "Eu acredito nisso, eu respondo por isso, porque, ao que parece, toda a questão humana consiste mesmo só em que o homem prove a si mesmo, a cada minuto, que é um homem, e não um pino! Ainda que com as próprias costelas, mas que prove; ainda que como um troglodita, mas que prove.", 0.95, 0.5],
        ["leitor", "E, depois disso, como não pecar, como não louvar que isso ainda não existe, e que o querer, por enquanto, ainda depende o diabo sabe de quê...", 0.95, 0.8],
        ["leitor", "Os senhores me gritam, se ainda me derem a honra do seu grito, que ninguém está tirando a minha vontade; que só estão cuidando de arranjar as coisas para que a minha vontade, por sua própria vontade, coincida com os meus interesses normais, com as leis da natureza e com a aritmética.", 0.95, 0.5],
        ["leitor", "Ah, senhores, que vontade própria vai haver, quando a coisa chegar à tabelinha e à aritmética, quando só estiver valendo o dois e dois são quatro? Dois e dois serão quatro mesmo sem a minha vontade. Isso lá é vontade própria!", 0.95, 1.5],
        ["comentarista", "A definição do homem como criatura de duas pernas, e ingrata, brinca com uma anedota antiga. Diógenes Laércio conta que Platão definiu o homem como um bípede sem penas, e que Diógenes, o cínico, depenou uma galinha e a levou à escola dizendo: eis o homem de Platão.", 1.0, 0.5],
        ["comentarista", "E o pino de realejo, a tecla de piano, volta muitas vezes. Uma leitura possível: para o narrador, ser previsível é deixar de ser pessoa. Daí a frase central do capítulo: o capricho nos conserva a individualidade, mesmo quando nos faz mal.", 1.0, 1.5],
    ]},
    9: {"titulo": "Dois e dois são cinco", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo nove.", 1.0, 0.8],
        ["comentarista", "O narrador admite que está brincando, mas avisa que brinca rangendo os dentes. E chega a uma das suas frases mais citadas: dois e dois são quatro já não é a vida, é o começo da morte.", 1.0, 1.0],
        ["leitor", "Capítulo nove.", 0.95, 0.8],
        ["leitor", "Senhores, eu estou brincando, é claro, e sei que a brincadeira não saiu boa; mas nem tudo se pode tomar por brincadeira. Talvez eu brinque rangendo os dentes. Senhores, as perguntas me torturam; respondam-nas para mim.", 0.95, 0.5],
        ["leitor", "Os senhores, por exemplo, querem desacostumar o homem dos velhos hábitos e corrigir a sua vontade, de acordo com as exigências da ciência e do bom senso. Mas como sabem que o homem não só pode, mas deve ser refeito assim?", 0.95, 0.5],
        ["leitor", "De onde concluem que o querer humano precisa tanto ser corrigido? Numa palavra, como sabem que essa correção vai mesmo trazer vantagem ao homem?", 0.95, 0.5],
        ["leitor", "E, para dizer tudo, por que estão tão seguros de que não ir contra as vantagens normais, garantidas pelos argumentos da razão e pela aritmética, é sempre vantajoso para o homem, e é uma lei para toda a humanidade?", 0.95, 0.5],
        ["leitor", "Por enquanto, isso é só uma suposição dos senhores. Suponhamos que seja uma lei da lógica; mas talvez não seja, de jeito nenhum, uma lei da humanidade.", 0.95, 0.8],
        ["leitor", "Os senhores talvez pensem que eu sou louco? Permitam-me explicar. Concordo: o homem é um animal sobretudo construtor, condenado a buscar um fim conscientemente e a se ocupar da arte da engenharia, isto é, a abrir caminho, eterna e incessantemente, seja para onde for.", 0.95, 0.5],
        ["leitor", "Mas talvez seja justamente por isso que às vezes lhe dá vontade de desviar para o lado: porque está condenado a abrir esse caminho;", 0.95, 0.5],
        ["leitor", "e talvez também porque, por mais tolo que seja o homem de ação, às vezes lhe vem à cabeça que o caminho quase sempre vai para onde for, e que o principal não é para onde vai, mas que ele ande,", 0.95, 0.5],
        ["leitor", "e que a criança bem-comportada, desprezando a arte da engenharia, não se entregue à ociosidade funesta, que, como se sabe, é a mãe de todos os vícios.", 0.95, 0.8],
        ["leitor", "O homem gosta de construir e de abrir estradas, isso é indiscutível. Mas por que ele também ama apaixonadamente a destruição e o caos? Isso é que eu queria que me dissessem!", 0.95, 0.5],
        ["leitor", "Não será, talvez, que ele ama tanto a destruição e o caos porque tem medo, por instinto, de alcançar o fim e terminar o edifício que constrói?", 0.95, 0.5],
        ["leitor", "Como vão saber? Talvez ele ame o edifício só de longe, e nunca de perto; talvez ele só goste de construí-lo, e não de morar nele, deixando-o depois para os animais domésticos, como as formigas, os carneiros, e assim por diante.", 0.95, 0.5],
        ["leitor", "As formigas, sim, são de outro gosto. Elas têm um edifício admirável desse tipo, eternamente indestrutível: o formigueiro. Pelo formigueiro as respeitáveis formigas começaram, e pelo formigueiro com certeza vão terminar, o que honra muito a sua constância e o seu senso positivo.", 0.95, 0.8],
        ["leitor", "Mas o homem é uma criatura leviana e de má aparência, e talvez, como o jogador de xadrez, só goste do processo de alcançar o objetivo, e não do próprio objetivo.", 0.95, 0.5],
        ["leitor", "E quem sabe, não se pode garantir, talvez todo o objetivo na terra a que a humanidade aspira consista só nessa continuidade do processo de alcançar, ou, em outras palavras, na própria vida, e não no objetivo propriamente dito,", 0.95, 0.5],
        ["leitor", "que, naturalmente, só pode ser dois e dois são quatro, isto é, uma fórmula; e dois e dois são quatro, senhores, já não é a vida, mas o começo da morte.", 0.95, 1.0],
        ["leitor", "Pelo menos, o homem sempre teve um certo medo desse dois e dois são quatro, e eu tenho medo até hoje. Suponhamos que o homem só faça procurar esse dois e dois são quatro, atravesse oceanos, sacrifique a vida nessa procura; mas encontrar, encontrar de verdade, juro por Deus, ele tem um certo medo.", 0.95, 0.5],
        ["leitor", "Pois sente que, quando encontrar, não haverá mais o que procurar. Os operários, terminado o trabalho, pelo menos recebem o dinheiro, vão para a taverna, depois param na delegacia; e aí está a ocupação da semana. Mas o homem, para onde vai?", 0.95, 0.5],
        ["leitor", "Ele gosta de alcançar, mas de ter alcançado, nem tanto, e isso, é claro, é terrivelmente engraçado. Numa palavra, o homem é feito de um jeito cômico; em tudo isso, evidentemente, há um trocadilho.", 0.95, 0.8],
        ["leitor", "Mas dois e dois são quatro é, mesmo assim, uma coisa insuportável. Dois e dois são quatro é, na minha opinião, só insolência. Dois e dois são quatro fica parado como um janota, de mãos na cintura, atravessado no seu caminho, e cospe.", 0.95, 0.5],
        ["leitor", "Concordo que dois e dois são quatro é uma coisa excelente; mas, se é para elogiar tudo, então dois e dois são cinco às vezes também é uma coisinha encantadora.", 0.95, 1.0],
        ["leitor", "E por que os senhores estão tão firme e solenemente convencidos de que só o normal e o positivo, numa palavra, só a prosperidade, é vantajosa para o homem? A razão não estará se enganando sobre as vantagens?", 0.95, 0.5],
        ["leitor", "Talvez o homem não ame só a prosperidade? Talvez ele ame, na mesma medida, o sofrimento? Talvez o sofrimento lhe seja tão vantajoso quanto a prosperidade? E o homem às vezes ama terrivelmente o sofrimento, até a paixão, e isso é um fato.", 0.95, 0.5],
        ["leitor", "Aqui nem é preciso consultar a história universal; perguntem a si mesmos, se forem homens e tiverem vivido um pouco. Quanto à minha opinião pessoal, amar só a prosperidade é até meio indecente. Bom ou ruim, quebrar alguma coisa de vez em quando também é muito agradável.", 0.95, 0.5],
        ["leitor", "Eu, aqui, não defendo propriamente o sofrimento, nem a prosperidade. Defendo... o meu capricho, e que ele me seja garantido quando eu precisar.", 0.95, 0.8],
        ["leitor", "O sofrimento, por exemplo, não é admitido nos vaudeviles, eu sei. No palácio de cristal, ele é impensável: o sofrimento é dúvida, é negação, e que palácio de cristal é esse, em que se pode duvidar?", 0.95, 0.5],
        ["leitor", "E, no entanto, tenho certeza de que o homem nunca vai renunciar ao sofrimento verdadeiro, isto é, à destruição e ao caos. O sofrimento é a única causa da consciência.", 0.95, 0.5],
        ["leitor", "Embora eu tenha declarado no começo que a consciência, para mim, é a maior infelicidade do homem, sei que o homem a ama e não a troca por satisfação nenhuma. A consciência, por exemplo, é infinitamente superior ao dois e dois.", 0.95, 0.5],
        ["leitor", "Depois do dois e dois, naturalmente, não vai sobrar nada, não só para fazer, mas nem para conhecer. Tudo o que se poderá fazer então é tapar os cinco sentidos e mergulhar na contemplação.", 0.95, 0.5],
        ["leitor", "Com a consciência, mesmo que o resultado seja o mesmo, isto é, também não haja nada a fazer, pelo menos às vezes se pode açoitar a si mesmo, e isso, mesmo assim, reanima. Ainda que seja retrógrado, é melhor do que nada.", 0.95, 1.5],
        ["comentarista", "O dois e dois são cinco que Orwell usaria em 1984 já aparece aqui, oitenta e cinco anos antes, mas com sinal trocado: no subsolo, é o direito de errar; em Orwell, é a mentira imposta pelo Estado.", 1.0, 0.5],
        ["comentarista", "Dentro da obra de Dostoiévski, este capítulo prepara uma cena famosa de Os Irmãos Karamázov, de 1880: o Grande Inquisidor, que acusa Cristo de ter dado aos homens uma liberdade pesada demais. Uma leitura possível: o homem do subsolo é o primeiro a dizer que prefere essa liberdade, mesmo que doa.", 1.0, 1.5],
    ]},
    10: {"titulo": "O galinheiro", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo dez.", 1.0, 0.8],
        ["comentarista", "Um capítulo curto, e com uma história de censura por trás. Vamos contá-la no fim.", 1.0, 1.0],
        ["leitor", "Capítulo dez.", 0.95, 0.8],
        ["leitor", "Os senhores acreditam no edifício de cristal, eternamente indestrutível, isto é, num edifício para o qual não se poderá nem mostrar a língua às escondidas, nem fazer uma figa no bolso.", 0.95, 0.5],
        ["leitor", "Pois eu, talvez, tenha medo desse edifício justamente porque ele é de cristal e eternamente indestrutível, e porque não se poderá nem mostrar a língua para ele às escondidas.", 0.95, 0.8],
        ["leitor", "Vejam: se, em vez de um palácio, houver um galinheiro, e começar a chover, eu talvez entre no galinheiro para não me molhar; mas, mesmo assim, não vou tomar o galinheiro por um palácio, por gratidão por ele ter me protegido da chuva.", 0.95, 0.5],
        ["leitor", "Os senhores riem, dizem até que, nesse caso, galinheiro e palacete dão no mesmo. Sim, eu respondo, se fosse preciso viver só para não se molhar.", 0.95, 0.8],
        ["leitor", "Mas o que fazer, se eu meti na cabeça que não se vive só para isso, e que, se é para viver, então que seja num palacete? Este é o meu querer, este é o meu desejo. Os senhores só vão raspá-lo de mim quando mudarem os meus desejos.", 0.95, 0.5],
        ["leitor", "Pois mudem, seduzam-me com outra coisa, deem-me outro ideal. Enquanto isso, não vou tomar o galinheiro por um palácio.", 0.95, 0.5],
        ["leitor", "Que seja até que o edifício de cristal é uma balela, que pelas leis da natureza ele nem está previsto, e que eu o inventei só por causa da minha própria tolice, de certos hábitos antiquados e irracionais da nossa geração.", 0.95, 0.5],
        ["leitor", "Mas que me importa que ele não esteja previsto? Não dá no mesmo, se ele existe nos meus desejos, ou melhor, existe enquanto existirem os meus desejos?", 0.95, 0.5],
        ["leitor", "Talvez os senhores riam de novo? Podem rir; aceito todas as zombarias, e mesmo assim não vou dizer que estou satisfeito quando tenho fome; mesmo assim sei que não vou me acalmar com um meio-termo, com um zero periódico sem fim, só porque ele existe pelas leis da natureza, e existe de verdade.", 0.95, 0.5],
        ["leitor", "Não vou aceitar como coroa dos meus desejos um prédio de alvenaria, com apartamentos para inquilinos pobres, com contrato de mil anos, e, por via das dúvidas, com a placa do dentista Wagenheim na porta.", 0.95, 0.5],
        ["leitor", "Destruam os meus desejos, apaguem os meus ideais, mostrem-me alguma coisa melhor, e eu sigo os senhores. Os senhores talvez digam que nem vale a pena se envolver; mas, nesse caso, eu posso responder o mesmo.", 0.95, 0.5],
        ["leitor", "Estamos raciocinando a sério; e, se não querem me honrar com a sua atenção, não vou fazer reverências. Eu tenho o subsolo.", 0.95, 1.0],
        ["leitor", "E, enquanto eu ainda vivo e desejo, que me seque a mão se eu levar um único tijolinho para um prédio desses! Não reparem que eu mesmo, há pouco, rejeitei o edifício de cristal só porque não se poderá provocá-lo com a língua.", 0.95, 0.5],
        ["leitor", "Não disse isso porque goste tanto de mostrar a língua. Talvez eu só estivesse com raiva porque, entre todos os edifícios dos senhores, até hoje não se encontra um para o qual não desse vontade de mostrar a língua.", 0.95, 0.5],
        ["leitor", "Ao contrário: eu deixaria que me cortassem a língua de vez, só por gratidão, se as coisas se arranjassem de modo que eu mesmo nunca mais tivesse vontade de mostrá-la.", 0.95, 0.5],
        ["leitor", "Que me importa que seja impossível arranjar isso, e que seja preciso se contentar com apartamentos? Por que, então, eu fui feito com desejos assim? Será que fui feito só para chegar à conclusão de que toda a minha construção é uma fraude? Será que todo o objetivo é esse? Não acredito.", 0.95, 1.0],
        ["leitor", "Mas, aliás, sabem de uma coisa: estou convencido de que gente como nós, do subsolo, precisa ser mantida de rédea curta. Embora seja capaz de ficar quarenta anos calado no subsolo, quando sai para a luz e desanda, aí fala, fala, fala...", 0.95, 1.5],
        ["comentarista", "A história de censura: em março de 1864, Dostoiévski escreveu ao irmão Mikhail reclamando que os censores tinham cortado justamente o trecho deste capítulo em que ele deduzia, de tudo isso, a necessidade da fé e de Cristo. Por isso o capítulo parece terminar de repente.", 1.0, 0.5],
        ["comentarista", "O texto cortado nunca foi recuperado, e Dostoiévski não o devolveu nas edições seguintes. O que sobrou é a pergunta, sem a resposta: será que fui feito só para descobrir que sou uma fraude? Não acredito.", 1.0, 0.5],
        ["comentarista", "E o prédio de apartamentos com contrato de mil anos é, segundo os comentadores, uma caricatura dos falanstérios de Fourier, as moradias coletivas que inspiravam os socialistas russos. O dentista Wagenheim, do capítulo quatro, volta na placa da porta.", 1.0, 1.5],
    ]},
    11: {"titulo": "A propósito da neve molhada", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo onze, o último da primeira parte.", 1.0, 0.8],
        ["comentarista", "Depois de dez capítulos de discussão, o narrador desmonta tudo o que disse, e anuncia a segunda parte.", 1.0, 1.0],
        ["leitor", "Capítulo onze.", 0.95, 0.8],
        ["leitor", "Conclusão final, senhores: é melhor não fazer nada! Melhor a inércia consciente! Portanto, viva o subsolo!", 0.95, 0.5],
        ["leitor", "Embora eu tenha dito que invejo o homem normal até a última gota de bile, nas condições em que o vejo não quero ser ele (embora, mesmo assim, não vá parar de invejá-lo. Não, não, o subsolo, em todo caso, é mais vantajoso!). Lá, pelo menos, pode-se...", 0.95, 0.5],
        ["leitor", "Ah! Mas aqui também estou mentindo! Minto, porque eu mesmo sei, como dois e dois, que o melhor não é de jeito nenhum o subsolo, mas outra coisa, completamente outra, de que tenho sede, mas que não consigo encontrar de jeito nenhum! Ao diabo o subsolo!", 0.95, 1.0],
        ["leitor", "Sabem o que seria até melhor? Se eu mesmo acreditasse em pelo menos alguma coisa de tudo o que escrevi agora. Juro-lhes, senhores, que não acredito em nenhuma, nenhuma palavrinha do que rabisquei!", 0.95, 0.5],
        ["leitor", "Quer dizer, talvez eu até acredite, mas, ao mesmo tempo, não se sabe por quê, sinto e desconfio que estou mentindo como um sapateiro.", 0.95, 0.8],
        ["leitor", "Então para que escreveu tudo isso? os senhores me dizem.", 0.95, 0.5],
        ["leitor", "Pois eu queria pôr os senhores quarenta anos sem nenhuma ocupação, e depois de quarenta anos vir ao subsolo visitá-los, para ver aonde chegaram. Será que se pode deixar um homem sozinho, sem ter o que fazer, por quarenta anos?", 0.95, 0.8],
        ["leitor", "E isso não é vergonhoso, e isso não é humilhante! talvez os senhores me digam, balançando a cabeça com desprezo. O senhor tem sede de vida, e resolve as questões da vida com uma confusão lógica.", 0.95, 0.5],
        ["leitor", "E como são insistentes, como são insolentes as suas tiradas, e ao mesmo tempo como o senhor tem medo! O senhor diz disparates e fica contente com eles; diz insolências, e ao mesmo tempo tem medo delas sem parar, e pede desculpas.", 0.95, 0.5],
        ["leitor", "Garante que não tem medo de nada, e ao mesmo tempo bajula a nossa opinião. Garante que range os dentes, e ao mesmo tempo faz graça para nos fazer rir.", 0.95, 0.5],
        ["leitor", "Sabe que as suas graças não têm graça, mas, evidentemente, está muito satisfeito com o valor literário delas. Talvez o senhor tenha mesmo sofrido, mas não respeita nem um pouco o seu sofrimento.", 0.95, 0.5],
        ["leitor", "No senhor há verdade, mas não há pudor; por vaidade mesquinha, o senhor leva a verdade para a vitrine, para a vergonha, para a feira...", 0.95, 0.5],
        ["leitor", "O senhor quer de fato dizer alguma coisa, mas, por medo, esconde a sua última palavra, porque não tem coragem de dizê-la, só tem insolência covarde.", 0.95, 0.5],
        ["leitor", "O senhor se gaba da consciência, mas só hesita, porque, embora a sua cabeça trabalhe, o seu coração está escurecido pela depravação, e sem um coração puro não haverá consciência plena e correta.", 0.95, 0.5],
        ["leitor", "E quanta insistência no senhor, como se oferece, como faz caretas! Mentira, mentira e mentira!", 0.95, 1.0],
        ["leitor", "É claro que todas essas palavras dos senhores fui eu mesmo que compus agora. Isso também vem do subsolo.", 0.95, 0.5],
        ["leitor", "Lá, por quarenta anos seguidos, eu escutei essas palavras dos senhores pela fresta. Eu mesmo as inventei; era só isso que se inventava. Não é de admirar que eu as tenha decorado e que tenham ganhado forma literária...", 0.95, 0.8],
        ["leitor", "Mas será, será que os senhores são de fato tão levianos a ponto de imaginar que eu vou publicar tudo isso, e ainda dar para os senhores lerem? E aqui está mais um problema para mim: para que, de fato, chamo os senhores de senhores, para que me dirijo aos senhores como se fossem mesmo leitores?", 0.95, 0.5],
        ["leitor", "Confissões como as que pretendo começar a expor não se publicam, nem se dão aos outros para ler. Mas vejam: me veio à cabeça uma fantasia, e quero realizá-la custe o que custar. É disto que se trata.", 0.95, 0.8],
        ["leitor", "Há nas lembranças de cada homem coisas que ele não revela a todos, mas só aos amigos. Há também coisas que ele não revela nem aos amigos, mas só a si mesmo, e ainda assim em segredo.", 0.95, 0.5],
        ["leitor", "Mas há, enfim, coisas que o homem tem medo de revelar até a si mesmo, e todo homem decente acumula uma boa quantidade delas. Quer dizer, até assim: quanto mais decente é o homem, mais delas ele tem.", 0.95, 0.5],
        ["leitor", "Pelo menos, eu mesmo só há pouco me decidi a relembrar certas aventuras minhas de antes, e até agora sempre as contornei, até com certa inquietação.", 0.95, 0.5],
        ["leitor", "Agora, que não só relembro, mas até me decidi a escrevê-las, agora quero justamente pôr à prova: será possível ser completamente franco ao menos consigo mesmo, e não ter medo de toda a verdade?", 0.95, 0.8],
        ["leitor", "Observo, a propósito: Heine afirma que autobiografias fiéis são quase impossíveis, e que o homem com certeza vai mentir sobre si mesmo. Na opinião dele, Rousseau, por exemplo, mentiu sobre si na sua confissão, e mentiu de propósito, por vaidade.", 0.95, 0.5],
        ["leitor", "Tenho certeza de que Heine tem razão; entendo muito bem como, às vezes, só por vaidade, se pode atribuir a si crimes inteiros, e compreendo muito bem até que tipo de vaidade pode ser essa.", 0.95, 0.5],
        ["leitor", "Mas Heine julgava um homem que se confessava diante do público. Eu escrevo só para mim, e declaro de uma vez por todas que, se escrevo como quem se dirige a leitores, é só para efeito, porque assim me é mais fácil escrever. É forma, só uma forma vazia; leitores eu nunca terei.", 0.95, 0.8],
        ["leitor", "(Corte: ele avisa que vai escrever sem ordem nem sistema, e discute consigo mesmo por que ainda assim se explica e se desculpa.)", 1.0, 0.6],
        ["leitor", "Mas há mais isto: para que, exatamente, eu quero escrever? Se não é para o público, eu poderia relembrar tudo de cabeça, sem passar para o papel.", 0.95, 0.5],
        ["leitor", "Pois é; mas no papel fica mais solene. Há nisso algo que impõe, haverá mais julgamento sobre mim mesmo, o estilo vai ganhar. Além disso, talvez eu de fato encontre alívio ao escrever.", 0.95, 0.5],
        ["leitor", "Hoje, por exemplo, uma lembrança antiga me oprime especialmente. Ela me veio com clareza há poucos dias, e desde então ficou comigo, como um motivo musical irritante, que não quer me largar. E, no entanto, é preciso me livrar dela.", 0.95, 0.5],
        ["leitor", "Tenho centenas de lembranças assim; mas, de vez em quando, uma delas se destaca entre as cem e oprime. Por algum motivo acredito que, se eu a escrever, ela vai me largar. Por que não tentar?", 0.95, 0.8],
        ["leitor", "Por fim: estou entediado, e não faço nada, sempre. E escrever, de fato, é como um trabalho. Dizem que o trabalho torna o homem bom e honesto. Pois aí está, pelo menos, uma chance.", 0.95, 1.0],
        ["leitor", "Hoje está caindo neve, quase molhada, amarela, turva. Ontem também caiu, e nesses dias também caiu. Acho que foi a propósito da neve molhada que eu me lembrei daquela história que agora não quer me largar.", 0.95, 0.5],
        ["leitor", "Então, que seja uma novela a propósito da neve molhada.", 0.95, 2.0],
        ["comentarista", "Com isso termina a Parte um. A Parte dois se chama justamente A propósito da neve molhada, e é outra coisa: não mais um monólogo, mas uma história, passada uns dezesseis anos antes, quando o narrador tinha vinte e quatro.", 1.0, 0.5],
        ["comentarista", "Sobre Heine: o poeta alemão escreveu nas suas Confissões, de 1854, que Rousseau mentiu nas dele. O narrador entra assim numa linha de livros de confissão, que vai de Santo Agostinho a Rousseau. A diferença, numa leitura possível, é que ele promete a verdade justamente porque diz que ninguém vai ler.", 1.0, 0.5],
        ["comentarista", "E repare que o leitor imaginado, os senhores, já foi desmascarado: o próprio narrador confessa que inventou as nossas objeções. Desde o primeiro capítulo, quem fala com ele somos nós, e somos uma invenção dele.", 1.0, 2.0],
    ]},
}

for n, c in CAPITULOS.items():
    m = {v: sum(len(t[1].split()) for t in c["trechos"] if t[0] == v) / 140 for v in VOZES}
    print(f"Capítulo {n}, {c['titulo']}: texto {m['leitor']:.1f} min, comentários {m['comentarista']:.1f} min.")

In [ ]:
# ── 6. Criar as duas vozes e ouvir (troque a semente se não gostar) ─────────
import hashlib, os, requests
from IPython.display import Audio, display

def gerar(texto, destino, **campos):
    arquivos = campos.pop("files", None)
    dados = {"text": texto, "language": "pt", "effect_preset": "raw", **campos}
    r = requests.post(f"{BASE}/generate", data=dados, files=arquivos, timeout=1800)
    if r.status_code != 200:
        raise SystemExit(f"Erro {r.status_code} em /generate: {r.text[:500]}\nLog: {LOG_PATH}")
    with open(destino, "wb") as f:
        f.write(r.content)

def chave(*partes):
    return hashlib.sha1(repr(partes).encode()).hexdigest()[:8]

REF = {}   # voz -> (wav de referência, texto)
for nome, v in VOZES.items():
    texto = next(max(falas, key=len) for _, c in sorted(CAPITULOS.items())
                 if (falas := [t[1] for t in c["trechos"] if t[0] == nome]))
    wav = os.path.join(PASTA_VOZES, f"{nome}_{chave(v, texto)}.wav")
    if not os.path.exists(wav):
        gerar(texto, wav, instruct=v["voz"], seed=v["semente"])
    REF[nome] = (wav, texto)
    print(f"{nome} ({v['voz']}, semente {v['semente']}): {texto}")
    display(Audio(wav))

In [ ]:
# ── 7. Narrar e montar cada capítulo (guiada e limpa) ───────────────────────
import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def narrar(n):
    pasta = os.path.join(LIVRO, f"cap{n:02d}", "trechos")
    os.makedirs(pasta, exist_ok=True)
    saida = []
    for i, (quem, texto, vel, pausa) in enumerate(CAPITULOS[n]["trechos"], 1):
        ref_wav, ref_texto = REF[quem]
        wav = os.path.join(pasta, f"{i:03d}_{quem}_{chave(os.path.basename(ref_wav), texto, vel)}.wav")
        if not os.path.exists(wav):
            with open(ref_wav, "rb") as ref:
                gerar(texto, wav, seed=VOZES[quem]["semente"], ref_text=ref_texto, speed=vel,
                      files={"ref_audio": ("ref.wav", ref, "audio/wav")})
            print(f"{n}.{i:03d} {quem}: {texto}")
        saida.append((quem, wav, pausa))
    return saida

def _nota(f, dur, vol, sr):
    t = np.arange(int(dur * sr)) / sr
    env = np.clip(t / 0.02, 0, 1) * np.exp(-4 * t)
    return (vol * (np.sin(2 * np.pi * f * t) + 0.2 * np.sin(4 * np.pi * f * t)) * env).astype("float32")

def sinal(sr, entra):
    # Duas notas curtas e baixas: sobem quando o comentário começa, descem quando acaba.
    fs = (440.0, 659.25) if entra else (659.25, 440.0)
    out = np.zeros(int(1.0 * sr), dtype="float32")
    for k, f in enumerate(fs):
        nota = _nota(f, 0.6, 0.06, sr)
        i = int(k * 0.18 * sr)
        out[i:i + len(nota)] += nota[:len(out) - i]
    return out

def exportar(audio, sr, pasta, nome):
    tmp = os.path.join(pasta, nome + ".wav")
    sf.write(tmp, audio, sr)
    mp3 = os.path.join(pasta, nome + ".mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100",
                    "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)
    os.remove(tmp)
    print(f"{nome}.mp3: {len(audio) / sr / 60:.1f} min")
    display(Audio(mp3))

def montar(n, trechos):
    sr = None
    guiada, limpa, anterior = [], [], "leitor"
    for quem, wav, pausa in trechos:
        a, s = sf.read(wav, dtype="float32", always_2d=True)
        sr = sr or s
        assert s == sr, f"taxa de amostragem diferente em {wav}"
        fala = a.mean(axis=1)
        silencio = np.zeros(int(pausa * sr), dtype="float32")
        if quem != anterior:
            guiada += [sinal(sr, entra=(quem == "comentarista")), np.zeros(int(0.3 * sr), dtype="float32")]
        guiada += [fala, silencio]
        if quem == "leitor":
            limpa += [fala, silencio]
        anterior = quem
    if anterior == "comentarista":
        guiada.append(sinal(sr, entra=False))
    um_seg = np.zeros(sr, dtype="float32")
    pasta = os.path.join(LIVRO, f"cap{n:02d}")
    exportar(np.concatenate([um_seg, *guiada, um_seg]), sr, pasta, f"cap{n:02d}_guiada")
    exportar(np.concatenate([um_seg, *limpa, um_seg]), sr, pasta, f"cap{n:02d}_limpa")

REFAZER = False   # True: narra de novo capítulos que já têm os dois MP3

for n in CAPITULOS_PARA_GERAR:
    print(f"\n=== Capítulo {n}: {CAPITULOS[n]['titulo']} ===")
    prontos = [os.path.join(LIVRO, f"cap{n:02d}", f"cap{n:02d}_{v}.mp3") for v in ("guiada", "limpa")]
    if not REFAZER and all(map(os.path.exists, prontos)):
        print("Já estava pronto (para refazer, use REFAZER = True).")
        continue
    montar(n, narrar(n))
print("\nPronto! Os arquivos estão no Drive, em", LIVRO)

# Se um trecho ficou ruim, apague o .wav dele na pasta capNN/trechos/ e rode esta célula de novo.